# Leadership and Management Book Recommendation System

## Notebook 02 — Web Scraping

### Project
Leadership and Management Book Recommendation System

### Notebook Purpose

This notebook collects book information from publicly accessible web pages to complement the Open Library API dataset collected in Notebook 01.

The API collection produced 1,000 raw search records representing 950 unique Open Library works. However, several variables required by the project have limited or incomplete availability through the API.

The web scraping stage therefore focuses on collecting complementary book information, particularly where available:

- Book title
- Author
- Description or synopsis
- Rating
- Rating count
- Price
- Currency
- Category
- Availability
- Product or book URL
- Cover image URL
- ISBN
- Publication information

The target is approximately 1,000 raw leadership and management related book records.

The scraped dataset will remain separate from the API datasets until the Data Integration stage.

## 1. Objective

The objectives of the web scraping stage are to:

1. Identify a suitable publicly accessible book source.
2. Inspect the website structure before large-scale collection.
3. Determine which metadata fields can be reliably extracted.
4. Develop a reproducible scraping workflow.
5. Apply controlled request pacing.
6. Collect approximately 1,000 raw leadership and management related book records where the source supports the target.
7. Preserve source URLs and collection provenance.
8. Save the raw scraped dataset without cleaning or deduplication.

The web scraping stage will complement rather than replace the Open Library API dataset.

## 2. Data Collection Principles

The following principles will guide the scraping process:

### Publicly Accessible Content

Only publicly accessible book information will be collected.

Login-protected, private, or restricted content will not be accessed.

### Controlled Request Frequency

Requests will be paced to avoid placing unnecessary load on the source website.

### Source Attribution

The source website and individual book URLs will be retained in the dataset.

### Raw Data Preservation

The raw scraped dataset will not be cleaned, deduplicated, normalized, or statistically imputed during this notebook.

### Missing Data

Unavailable information will remain missing rather than being fabricated or inferred.

### Reproducibility

The scraping methodology, selectors, pagination logic, and collection process will be documented so the workflow can be reproduced and audited.

## 3. Scraping Workflow

The web scraping process will follow these stages:

1. Select and validate the source website.
2. Inspect the website structure.
3. Test access to a single page.
4. Identify book containers and HTML selectors.
5. Extract a small sample.
6. Inspect individual book pages.
7. Identify additional metadata.
8. Test pagination.
9. Validate extracted fields.
10. Scale the collection.
11. Perform a preliminary data quality assessment.
12. Save the raw scraped dataset.
13. Document collection limitations.

## 4. Import Libraries

### Purpose

The notebook imports libraries for HTTP requests, HTML parsing, controlled request timing, structured data handling, path management, and browser automation.

The initial scraping tests will use Requests and BeautifulSoup.

Selenium is also imported so that browser automation can be used if the selected website requires JavaScript rendering or interactive navigation.

In [1]:
import time
import pathlib

import requests
import pandas as pd

from bs4 import BeautifulSoup

# Selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import (
    NoSuchElementException,
    ElementClickInterceptedException
)

print("Libraries imported successfully.")

Libraries imported successfully.


## 5. Confirm Project Paths

### Purpose

The project paths are defined so that scraped data can be saved consistently within the existing repository structure.

Raw web scraping outputs will be stored in:

`data/raw/`

No files from the API collection will be overwritten.

In [2]:
project_path = pathlib.Path.cwd().parent

raw_data_path = project_path / "data" / "raw"

print("Project path:")
print(project_path)

print("\nRaw data path:")
print(raw_data_path)

print("\nRaw data path exists:")
print(raw_data_path.exists())

Project path:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System

Raw data path:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/raw

Raw data path exists:
True


## 6. Initial Scraping Source

The initial scraping workflow will be tested using Books to Scrape.

Website:

`https://books.toscrape.com/`

The website is designed for web scraping practice and provides structured book catalogue pages and individual product pages.

Potentially available variables include:

- Title
- Price
- Currency
- Availability
- Star rating
- Category
- Product description
- UPC
- Product type
- Tax information
- Number of reviews
- Product URL
- Cover image URL

### Source Evaluation Principle

The website will initially be used to validate the scraping methodology.

Before scaling collection, the catalogue will be evaluated to determine whether it contains sufficient leadership and management related books for the project's subject scope.

If the source does not provide adequate domain coverage, an alternative or additional publicly accessible source will be selected.

## 7. Test Website Access

### Purpose

Before developing extraction logic, the website will be tested to confirm that the catalogue page is publicly accessible through a standard HTTP request.

A successful HTTP response does not by itself guarantee that the complete site should be scraped. It only confirms technical access to the test page.

In [3]:
base_url = "https://books.toscrape.com/"

headers = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome Safari"
    )
}

response = requests.get(
    base_url,
    headers=headers,
    timeout=30
)

print("Status code:", response.status_code)

Status code: 200


In [4]:
soup = BeautifulSoup(
    response.text,
    "html.parser"
)

print("Page title:")
print(soup.title.text.strip())

Page title:
All products | Books to Scrape - Sandbox


In [5]:
book_cards = soup.select(
    "article.product_pod"
)

print(
    "Book cards found:",
    len(book_cards)
)

Book cards found: 20


In [6]:
first_book = book_cards[0]

print(
    first_book.prettify()[:3000]
)

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [7]:
title_element = first_book.select_one(
    "h3 a"
)

price_element = first_book.select_one(
    ".price_color"
)

availability_element = first_book.select_one(
    ".availability"
)

rating_element = first_book.select_one(
    "p.star-rating"
)

print(
    "Title:",
    title_element.get("title")
)

print(
    "Price:",
    price_element.get_text(strip=True)
)

print(
    "Availability:",
    availability_element.get_text(
        " ",
        strip=True
    )
)

print(
    "Rating classes:",
    rating_element.get("class")
)

Title: A Light in the Attic
Price: Â£51.77
Availability: In stock
Rating classes: ['star-rating', 'Three']


## 10. Extract Individual Book URL

### Purpose

The catalogue page contains summary-level book information, while additional metadata is available on each individual product page.

The first book URL will be extracted and converted from a relative URL into a complete URL.

This allows the structure of an individual book page to be inspected before developing the complete scraper.

In [8]:
from urllib.parse import urljoin

relative_book_url = title_element.get("href")

book_url = urljoin(
    base_url,
    relative_book_url
)

print("Relative URL:")
print(relative_book_url)

print("\nComplete book URL:")
print(book_url)

Relative URL:
catalogue/a-light-in-the-attic_1000/index.html

Complete book URL:
https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


## 11. Test Individual Book Page

### Purpose

The individual product page will be requested to determine whether additional metadata can be collected reliably.

Only one book is tested at this stage.

In [9]:
book_response = requests.get(
    book_url,
    headers=headers,
    timeout=30
)

print(
    "Book page status code:",
    book_response.status_code
)

book_soup = BeautifulSoup(
    book_response.text,
    "html.parser"
)

print(
    "Book page title:",
    book_soup.title.text.strip()
)

Book page status code: 200
Book page title: A Light in the Attic | Books to Scrape - Sandbox


## 12. Inspect Product Information

### Purpose

The product information table will be inspected to identify structured metadata available for each book.

Potential fields include:

- UPC
- Product type
- Price excluding tax
- Price including tax
- Tax
- Availability
- Number of reviews

These fields may complement the metadata collected through the Open Library API.

In [10]:
product_table = book_soup.select_one(
    "table.table.table-striped"
)

print(
    product_table.prettify()[:3000]
)

<table class="table table-striped">
 <tr>
  <th>
   UPC
  </th>
  <td>
   a897fe39b1053632
  </td>
 </tr>
 <tr>
  <th>
   Product Type
  </th>
  <td>
   Books
  </td>
 </tr>
 <tr>
  <th>
   Price (excl. tax)
  </th>
  <td>
   Â£51.77
  </td>
 </tr>
 <tr>
  <th>
   Price (incl. tax)
  </th>
  <td>
   Â£51.77
  </td>
 </tr>
 <tr>
  <th>
   Tax
  </th>
  <td>
   Â£0.00
  </td>
 </tr>
 <tr>
  <th>
   Availability
  </th>
  <td>
   In stock (22 available)
  </td>
 </tr>
 <tr>
  <th>
   Number of reviews
  </th>
  <td>
   0
  </td>
 </tr>
</table>



In [11]:
product_information = {}

table_rows = product_table.select("tr")

for row in table_rows:
    
    field = row.select_one("th")
    value = row.select_one("td")
    
    if field and value:
        product_information[
            field.get_text(strip=True)
        ] = value.get_text(strip=True)

product_information

{'UPC': 'a897fe39b1053632',
 'Product Type': 'Books',
 'Price (excl. tax)': 'Â£51.77',
 'Price (incl. tax)': 'Â£51.77',
 'Tax': 'Â£0.00',
 'Availability': 'In stock (22 available)',
 'Number of reviews': '0'}

## 13. Extract Book Category

### Purpose

The website breadcrumb identifies the catalogue category assigned to the book.

Category information is particularly important for determining whether Books to Scrape contains sufficient leadership and management related material for the project.

In [12]:
breadcrumb_items = book_soup.select(
    "ul.breadcrumb li a"
)

for item in breadcrumb_items:
    print(
        item.get_text(strip=True),
        "->",
        item.get("href")
    )

Home -> ../../index.html
Books -> ../category/books_1/index.html
Poetry -> ../category/books/poetry_23/index.html


In [13]:
category = None

if len(breadcrumb_items) >= 3:
    category = breadcrumb_items[-1].get_text(
        strip=True
    )

print("Category:", category)

Category: Poetry


## 14. Extract Product Description

### Purpose

Product descriptions provide textual information that may support NLP, clustering, and content-based recommendations.

Description availability will therefore be tested on the individual product page.

In [14]:
description = None

description_heading = book_soup.find(
    "div",
    id="product_description"
)

if description_heading:
    
    description_element = (
        description_heading.find_next_sibling("p")
    )
    
    if description_element:
        description = (
            description_element.get_text(
                " ",
                strip=True
            )
        )

print("Description:")
print(description)

Description:
It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,And your cradle, too?Baby, I think someone down here'sGot it in for you. Shel, you 

## 15. Extract Cover Image URL

### Purpose

The full cover image URL will be collected for potential use in the Streamlit recommendation application and exploratory book displays.

Only the image URL will be stored during data collection. Image files will not be downloaded at this stage.

In [15]:
cover_element = book_soup.select_one(
    ".item.active img"
)

cover_url = None

if cover_element:
    
    cover_url = urljoin(
        book_url,
        cover_element.get("src")
    )

print("Cover URL:")
print(cover_url)

Cover URL:
https://books.toscrape.com/media/cache/fe/72/fe72f0532301ec28892ae79a629a293c.jpg


## 16. Extract Star Rating

### Purpose

The website represents star ratings through CSS classes rather than directly displaying the numerical value.

The raw rating class will be retained, while a simple mapping will also convert the rating into a numerical value for later analysis.

In [16]:
rating_element = book_soup.select_one(
    "p.star-rating"
)

rating_class = None
rating_value = None

rating_mapping = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

if rating_element:
    
    rating_classes = rating_element.get(
        "class",
        []
    )
    
    for rating_name, numeric_value in rating_mapping.items():
        
        if rating_name in rating_classes:
            
            rating_class = rating_name
            rating_value = numeric_value
            break

print("Rating class:", rating_class)
print("Rating value:", rating_value)

Rating class: Three
Rating value: 3


## 17. Single-Book Extraction Test

### Purpose

The successfully identified fields will be combined into a single structured record.

This test establishes the preliminary schema for the scraped dataset before pagination or large-scale collection is attempted.

In [17]:
single_book_record = {
    "title": title_element.get("title"),
    "category": category,
    "rating_class": rating_class,
    "rating_value": rating_value,
    "price_raw": price_element.get_text(
        strip=True
    ),
    "availability_summary": (
        availability_element.get_text(
            " ",
            strip=True
        )
    ),
    "description": description,
    "upc": product_information.get("UPC"),
    "product_type": product_information.get(
        "Product Type"
    ),
    "price_excl_tax_raw": (
        product_information.get(
            "Price (excl. tax)"
        )
    ),
    "price_incl_tax_raw": (
        product_information.get(
            "Price (incl. tax)"
        )
    ),
    "tax_raw": product_information.get("Tax"),
    "availability_detail": (
        product_information.get(
            "Availability"
        )
    ),
    "number_of_reviews": (
        product_information.get(
            "Number of reviews"
        )
    ),
    "cover_url": cover_url,
    "book_url": book_url,
    "source": "Books to Scrape"
}

single_book_df = pd.DataFrame(
    [single_book_record]
)

single_book_df.T

,0
title,A Light in the Attic
category,Poetry
rating_class,Three
rating_value,3
price_raw,Â£51.77
availability_summary,In stock
description,It's hard to imagine a world without A Light i...
upc,a897fe39b1053632
product_type,Books
price_excl_tax_raw,Â£51.77


## 18. Inspect Website Categories

### Purpose

The initial scraping test confirmed that Books to Scrape can provide structured book metadata.

However, the project specifically requires leadership and management related books.

Before scaling the scraper, the website's complete category structure will be inspected to determine whether the catalogue contains categories sufficiently aligned with the project's subject domain.

This validation prevents the collection of a large general-purpose book dataset that would not adequately address the business problem.

In [18]:
category_links = soup.select(
    ".side_categories ul.nav-list ul li a"
)

categories = []

for category_link in category_links:

    category_name = category_link.get_text(
        strip=True
    )

    category_url = urljoin(
        base_url,
        category_link.get("href")
    )

    categories.append({
        "category": category_name,
        "category_url": category_url
    })

categories_df = pd.DataFrame(categories)

print(
    "Number of categories:",
    len(categories_df)
)

categories_df

Number of categories: 50


,category,category_url
0,Travel,https://books.toscrape.com/catalogue/category/...
1,Mystery,https://books.toscrape.com/catalogue/category/...
2,Historical Fiction,https://books.toscrape.com/catalogue/category/...
3,Sequential Art,https://books.toscrape.com/catalogue/category/...
4,Classics,https://books.toscrape.com/catalogue/category/...
5,Philosophy,https://books.toscrape.com/catalogue/category/...
6,Romance,https://books.toscrape.com/catalogue/category/...
7,Womens Fiction,https://books.toscrape.com/catalogue/category/...
8,Fiction,https://books.toscrape.com/catalogue/category/...
9,Childrens,https://books.toscrape.com/catalogue/category/...


## 19. Evaluate Category Relevance

### Purpose

The available categories will be searched for terms associated with the project's leadership and management scope.

This is an initial screening step only.

The absence of explicit leadership or management categories would indicate that Books to Scrape is better suited as a technical scraping demonstration source than as the primary domain-specific data source.

In [19]:
relevance_terms = [
    "leadership",
    "management",
    "business",
    "economics",
    "psychology",
    "self help",
    "self-help",
    "productivity",
    "entrepreneur",
    "communication",
    "career"
]

relevant_categories = categories_df[
    categories_df["category"]
    .str.lower()
    .apply(
        lambda category:
        any(
            term in category
            for term in relevance_terms
        )
    )
]

relevant_categories

,category,category_url
24,Psychology,https://books.toscrape.com/catalogue/category/...
33,Business,https://books.toscrape.com/catalogue/category/...
39,Self Help,https://books.toscrape.com/catalogue/category/...


In [20]:
first_category_url = (
    categories_df.loc[
        0,
        "category_url"
    ]
)

category_response = requests.get(
    first_category_url,
    headers=headers,
    timeout=30
)

print(
    "Status code:",
    category_response.status_code
)

category_soup = BeautifulSoup(
    category_response.text,
    "html.parser"
)

print(
    "Category:",
    categories_df.loc[0, "category"]
)

print(
    "Books visible on first page:",
    len(
        category_soup.select(
            "article.product_pod"
        )
    )
)

Status code: 200
Category: Travel
Books visible on first page: 11


In [21]:
pager = category_soup.select_one(
    "ul.pager"
)

if pager:
    print(
        pager.get_text(
            " ",
            strip=True
        )
    )
else:
    print(
        "No pagination element found."
    )

No pagination element found.


## 21. Preliminary Source Assessment

The single-book extraction test demonstrated that Books to Scrape is technically suitable for practicing and validating the project's web scraping methodology.

The website provides structured access to:

- Title
- Category
- Star rating
- Price
- Availability
- Product description
- UPC
- Product type
- Tax information
- Number of reviews
- Cover image URL
- Product URL

The first tested book was categorized as Poetry, demonstrating that the website represents a general book catalogue rather than a dedicated leadership and management catalogue.

Consequently, the website will not automatically be treated as the project's primary scraping source.

The complete category structure must first be evaluated to determine whether sufficient leadership and management related records can be identified.

## 22. Source Selection Decision

The Books to Scrape evaluation confirmed that the website is technically suitable for demonstrating web scraping methods but is not sufficiently aligned with the project's subject domain.

The website contains 50 book categories. Only a small subset, including Business, Psychology, and Self Help, has potential relevance to leadership and management.

Dedicated Leadership and Management categories are not available.

Collecting the complete catalogue would therefore introduce a substantial number of irrelevant books from categories such as fiction, romance, poetry, fantasy, horror, and other unrelated subjects.

### Decision

Books to Scrape will not be used as the primary source for the approximately 1,000-book web scraping dataset.

The source evaluation remains documented in this notebook because it demonstrates:

- website accessibility testing;
- HTML structure inspection;
- catalogue-level extraction;
- individual product-page extraction;
- category evaluation;
- source suitability assessment; and
- evidence-based source selection.

A more domain-specific book catalogue will now be evaluated.

### Candidate Source

Hugendubel's Management catalogue has been identified as a candidate because it contains a large collection of management-related books and provides sufficient catalogue depth for the project's target sample.

The candidate source will first undergo a small technical and structural test before any large-scale scraping is performed.

## 23. Test Candidate Management Catalogue

### Purpose

The candidate Management catalogue will be tested before developing extraction logic.

The objective of this step is to determine:

1. whether the public catalogue page can be accessed through a standard HTTP request;
2. whether book information is present in the returned HTML;
3. whether JavaScript rendering is required; and
4. whether Requests and BeautifulSoup are sufficient or Selenium is necessary.

No large-scale data collection will be performed during this test.

In [22]:
hugendubel_url = (
    "https://www.hugendubel.de/de/category/"
    "68972/management.html"
)

hugendubel_response = requests.get(
    hugendubel_url,
    headers=headers,
    timeout=30
)

print(
    "Status code:",
    hugendubel_response.status_code
)

print(
    "Response length:",
    len(hugendubel_response.text)
)

print(
    "Content type:",
    hugendubel_response.headers.get(
        "Content-Type"
    )
)

Status code: 503
Response length: 30668
Content type: text/html


## 25. Candidate Source Access Result

The Hugendubel Management catalogue was evaluated as a potential domain-specific source for the web scraping dataset.

The HTTP test returned:

- HTTP status code: 503
- Response type: text/html
- Response length: 30,668 characters

### Interpretation

The server did not return a normal successful HTTP response to the automated request.

A 503 response may indicate temporary service unavailability or automated-access protection. Therefore, the project will not attempt to circumvent the website's access controls.

Although the catalogue is relevant to the leadership and management domain, it is not suitable for the current Requests and BeautifulSoup collection workflow.

### Decision

Hugendubel will not be used as the primary scraping source under the current methodology.

The project will evaluate another publicly accessible source that provides leadership and management related book information without requiring circumvention of access restrictions.

## 26. Evaluate Alternative Management Catalogue

### Purpose

Following the unsuccessful HTTP access test for the previous candidate source, another domain-specific management book catalogue will be evaluated.

The initial test will only determine whether the publicly accessible catalogue can be retrieved through a standard HTTP request.

No large-scale scraping will be performed until accessibility and HTML structure have been validated.

In [23]:
thalia_url = (
    "https://www.thalia.de/"
    "kategorie/management-2397/"
)

thalia_response = requests.get(
    thalia_url,
    headers=headers,
    timeout=30
)

print(
    "Status code:",
    thalia_response.status_code
)

print(
    "Response length:",
    len(thalia_response.text)
)

print(
    "Content type:",
    thalia_response.headers.get(
        "Content-Type"
    )
)

Status code: 403
Response length: 12306
Content type: text/html; charset=UTF-8


## 27. Alternative Source Access Result

The Thalia Management catalogue was evaluated as another potential domain-specific source.

The HTTP test returned:

- HTTP status code: 403
- Response type: text/html; charset=UTF-8
- Response length: 12,306 characters

### Interpretation

HTTP status code 403 indicates that the server received the request but refused access.

The project will not attempt to bypass or circumvent the website's access restrictions.

### Decision

Thalia will not be used as the primary scraping source under the current methodology.

This source evaluation demonstrates that domain relevance alone is insufficient when selecting a web scraping source. The source must also support ethical, technically reproducible, and publicly accessible data collection.

## 28. Final Candidate Source Strategy

Following the initial source evaluations, the project identified three major constraints:

1. Books to Scrape is technically accessible but insufficiently focused on leadership and management.
2. Hugendubel provides strong domain relevance but returned HTTP 503 during automated access testing.
3. Thalia provides strong domain relevance but returned HTTP 403 during automated access testing.

The project will not attempt to circumvent website access restrictions.

### Next Candidate: Bookshop.org

Bookshop.org will now be evaluated as an independent commercial book source.

Publicly accessible leadership and management collections contain relevant books and potentially provide complementary commercial metadata including:

- Title
- Author
- Book format
- Current price
- List price
- Product URL
- Cover information

The source will first undergo technical and structural validation before large-scale collection is considered.

In [24]:
bookshop_url = (
    "https://bookshop.org/lists/"
    "best-books-on-leadership-and-management"
)

bookshop_response = requests.get(
    bookshop_url,
    headers=headers,
    timeout=30
)

print(
    "Status code:",
    bookshop_response.status_code
)

print(
    "Response length:",
    len(bookshop_response.text)
)

print(
    "Content type:",
    bookshop_response.headers.get(
        "Content-Type"
    )
)

Status code: 403
Response length: 5836
Content type: text/html; charset=UTF-8


## 30. Bookshop.org Access Result

Bookshop.org was evaluated as an additional commercial source for leadership and management books.

The HTTP test returned:

- HTTP status code: 403
- Response type: text/html; charset=UTF-8
- Response length: 5,836 characters

### Interpretation

The server received the automated HTTP request but refused access.

The project will not attempt to circumvent the website's access controls.

### Decision

Bookshop.org will not be used as the primary web scraping source.

Following multiple commercial-source evaluations, the project will shift toward an openly accessible catalogue that supports reproducible academic data collection.

In [25]:
import time

start_time = time.time()

try:
    response_test = requests.get(
        bookshop_url,
        headers=headers,
        timeout=60
    )

    elapsed_time = time.time() - start_time

    print("Status code:", response_test.status_code)
    print("Response time:", round(elapsed_time, 2), "seconds")
    print("Response length:", len(response_test.text))

except requests.exceptions.Timeout:
    print("The request timed out.")

except requests.exceptions.RequestException as error:
    print("Request error:", error)

Status code: 403
Response time: 0.1 seconds
Response length: 5857


## 30. Bookshop.org Access Result

Bookshop.org was evaluated as an additional commercial source for leadership and management books.

The initial HTTP request returned:

- HTTP status code: 403
- Response type: text/html; charset=UTF-8
- Response length: approximately 5,800 characters

A second diagnostic request was performed with the request timeout increased from 30 seconds to 60 seconds.

The diagnostic request returned:

- HTTP status code: 403
- Response time: 0.1 seconds
- Response length: 5,857 characters

### Interpretation

The response was returned almost immediately.

Therefore, the unsuccessful request was not caused by the configured timeout. The server actively returned an HTTP 403 response to the automated request.

Increasing the timeout would not resolve this type of response.

### Decision

The project will not attempt to circumvent the website's access controls.

Bookshop.org will not be used as the primary scraping source under the current Requests and BeautifulSoup methodology.

## 31. Selenium Browser Access Test

### Purpose

The previous Requests test returned HTTP 403 almost immediately, confirming that the issue was not caused by the request timeout.

Some modern websites rely heavily on browser-side JavaScript and may not provide the same content to a direct HTTP request that they provide through a normal web browser.

A standard Selenium-controlled Chrome browser will therefore be tested.

No stealth drivers, CAPTCHA bypasses, proxy rotation, fingerprint modification, or other access-control circumvention techniques will be used.

In [26]:
driver = webdriver.Chrome(
    service=Service(
        ChromeDriverManager().install()
    )
)

driver.get(bookshop_url)

time.sleep(5)

print("Page title:")
print(driver.title)

print("\nCurrent URL:")
print(driver.current_url)

print("\nPage source length:")
print(len(driver.page_source))

Page title:
best books on leadership and management - Bookshop.org

Current URL:
https://bookshop.org/lists/best-books-on-leadership-and-management

Page source length:
321076


In [27]:
page_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(
    page_text[:3000]
)

We've updated our Privacy Policy - effective June 6, 2026. We've made changes to how we share data with advertising and analytics partners, and expanded your rights under applicable state privacy laws. Please review the updated policy before continuing. You may change your settings at any time or accept the default settings. You may close this banner to continue with only essential cookies. Privacy Policy
Storage Preferences
Targeted Advertising
Personalization
Analytics
Save
Accept All
Reject Non-Essential
You appear to be in the United Kingdom 🇬🇧 Click here to switch to our UK website.
@Lynxotic
Powered by
Bookshop.org
Choose a Bookstore
Sign in
New Books
Ebooks
Best Sellers
Pre-Order Offers
Kids
Fiction
Nonfiction
YA
Games & Puzzles
Stationery & Gifts
Gift Cards
Offers
All orders support local bookstores. $50,634,846.31 raised so far!
best books on leadership and management
By @Lynxotic
best books on leadership and management
The Book on Leadership
John F MacArthur
Paperback
$18.63


In [28]:
driver.quit()

## 32. Selenium Access Result

The Bookshop.org leadership and management collection was tested using a standard Selenium-controlled Chrome browser.

### Results

The browser successfully loaded the public webpage.

Observed results:

- Page title: `best books on leadership and management - Bookshop.org`
- Public catalogue content successfully rendered
- Page source length: approximately 321,000 characters
- Leadership and management book records were visible in the rendered page

The rendered content included:

- Book title
- Author
- Book format
- Current price
- List price where available
- Availability status

Examples of visible titles included:

- The Book on Leadership
- Leadership
- The Five Dysfunctions of a Team
- Leadership 2.0
- The Heart of Business
- Authentic Leadership
- Leadership and the One Minute Manager
- Quiet Leadership
- Principle Centered Leadership

### Interpretation

The earlier HTTP 403 response was not caused by the configured request timeout.

However, the public webpage loads normally through a standard browser and its book information is rendered successfully.

Therefore, Selenium is technically appropriate for this source because the project requires browser-rendered content.

No CAPTCHA bypass, stealth driver, proxy rotation, fingerprint modification, or other access-control circumvention technique is being used.

### Decision

Bookshop.org will proceed to structural evaluation as a candidate web scraping source.

Before large-scale collection, the HTML structure, book containers, product URLs, metadata availability, and catalogue scalability will be evaluated.

## 33. Inspect Rendered Book Links

### Purpose

The rendered page contains multiple leadership and management books.

Before developing extraction logic, the page links will be inspected to identify the URL structure used for individual book product pages.

This avoids relying on assumed CSS selectors.

In [31]:
driver = webdriver.Chrome(
    service=Service(
        ChromeDriverManager().install()
    )
)

driver.get(bookshop_url)

time.sleep(5)

print("Page title:")
print(driver.title)

print("\nCurrent URL:")
print(driver.current_url)

print("\nPage source length:")
print(len(driver.page_source))

Page title:
best books on leadership and management - Bookshop.org

Current URL:
https://bookshop.org/lists/best-books-on-leadership-and-management

Page source length:
321200


In [32]:
all_links = driver.find_elements(
    By.TAG_NAME,
    "a"
)

print(
    "Total links found:",
    len(all_links)
)

Total links found: 264


In [33]:
print(
    "Selenium session ID:",
    driver.session_id
)

print(
    "Current page:",
    driver.title
)

Selenium session ID: c08b2cea001db2ca8c7e18e512ee1719
Current page: best books on leadership and management - Bookshop.org


In [34]:
all_links = driver.find_elements(
    By.TAG_NAME,
    "a"
)

print(
    "Total links found:",
    len(all_links)
)

Total links found: 264


In [35]:
link_records = []

for link in all_links:

    try:
        link_text = link.text.strip()
        link_href = link.get_attribute(
            "href"
        )

        if link_text and link_href:

            link_records.append({
                "text": link_text,
                "href": link_href
            })

    except Exception:
        pass

links_df = pd.DataFrame(
    link_records
)

print(
    "Links extracted:",
    len(links_df)
)

links_df.head(30)

Links extracted: 79


,text,href
0,Privacy Policy,https://bookshop.org/info/privacy-notice
1,Storage Preferences,https://bookshop.org/lists/best-books-on-leade...
2,Click here to switch,https://uk.bookshop.org/
3,@Lynxotic,https://bookshop.org/shop/cherrybooks
4,Powered by\nBookshop.org,https://bookshop.org/
5,Choose a Bookstore,https://bookshop.org/pages/bookstores
6,Sign in,https://bookshop.org/login
7,New Books,https://bookshop.org/lists/new-books
8,Ebooks,https://bookshop.org/ebooks
9,Best Sellers,https://bookshop.org/categories/m/popular-books


## 34. Identify Individual Book Product Links

### Purpose

The rendered Bookshop.org page contains navigation, account, category, and book-product links.

Inspection of the extracted hyperlinks shows that individual books use the URL pattern:

`https://bookshop.org/p/books/...`

This pattern will be used to distinguish book records from unrelated navigation links.

The resulting product links will provide the foundation for subsequent book-level metadata extraction.

In [36]:
book_links_df = links_df[
    links_df["href"]
    .str.contains(
        "bookshop.org/p/books/",
        case=False,
        na=False
    )
].copy()

book_links_df = (
    book_links_df
    .drop_duplicates(
        subset="href"
    )
    .reset_index(drop=True)
)

print(
    "Unique book product links:",
    len(book_links_df)
)

book_links_df.head(20)

Unique book product links: 30


,text,href
0,The Book on Leadership,https://bookshop.org/p/books/the-book-on-leade...
1,It Worked for Me,https://bookshop.org/p/books/it-worked-for-me-...
2,Leadership,https://bookshop.org/p/books/leadership-in-tur...
3,The Five Dysfunctions of a Team,https://bookshop.org/p/books/the-five-dysfunct...
4,Leadership 2.0,https://bookshop.org/p/books/leadership-2-0-dr...
5,Culturally Responsive School Leadership,https://bookshop.org/p/books/culturally-respon...
6,Leadership From Below,https://bookshop.org/p/books/leadership-from-b...
7,The Five Graces of Life and Leadership,https://bookshop.org/p/books/the-five-graces-o...
8,On Leadership,https://bookshop.org/p/books/on-leadership-joh...
9,The Heart of Business,https://bookshop.org/p/books/the-heart-of-busi...


In [37]:
book_links_df.tail(20)

,text,href
10,Trust-Based Leadership,https://bookshop.org/p/books/trust-based-leade...
11,No Bullsh!t Leadership,https://bookshop.org/p/books/no-bullsh-t-leade...
12,Glue,https://bookshop.org/p/books/glue-a-leadership...
13,Transformed Leadership,https://bookshop.org/p/books/transformed-leade...
14,Leadership Is an Art,https://bookshop.org/p/books/leadership-is-an-...
15,Permission to Glow,https://bookshop.org/p/books/permission-to-glo...
16,Leadership Is Leadership,https://bookshop.org/p/books/leadership-is-lea...
17,Leadership Presence,https://bookshop.org/p/books/leadership-presen...
18,Unmanageable,https://bookshop.org/p/books/unmanageable-lead...
19,Authentic Leadership,https://bookshop.org/p/books/authentic-leaders...


In [38]:
print(
    "First book:",
    book_links_df.iloc[0]["text"]
)

print(
    "Last book:",
    book_links_df.iloc[-1]["text"]
)

First book: The Book on Leadership
Last book: The Leadership Secrets of Colin Powell


In [39]:
known_books = book_links_df[
    book_links_df["text"]
    .str.contains(
        "Five Dysfunctions|Principle Centered Leadership",
        case=False,
        na=False,
        regex=True
    )
]

known_books

,text,href
3,The Five Dysfunctions of a Team,https://bookshop.org/p/books/the-five-dysfunct...
26,Principle Centered Leadership,https://bookshop.org/p/books/principle-centere...


In [40]:
initial_height = driver.execute_script(
    "return document.body.scrollHeight"
)

print(
    "Initial page height:",
    initial_height
)

Initial page height: 4557


In [41]:
driver.execute_script(
    "window.scrollTo(0, document.body.scrollHeight);"
)

time.sleep(3)

new_height = driver.execute_script(
    "return document.body.scrollHeight"
)

print(
    "Page height after scroll:",
    new_height
)

print(
    "Height changed:",
    new_height != initial_height
)

Page height after scroll: 4557
Height changed: False


In [42]:
all_links_after_scroll = driver.find_elements(
    By.TAG_NAME,
    "a"
)

book_urls_after_scroll = []

for link in all_links_after_scroll:

    try:
        href = link.get_attribute(
            "href"
        )

        if (
            href
            and "bookshop.org/p/books/" in href
        ):
            book_urls_after_scroll.append(
                href
            )

    except Exception:
        pass

unique_book_urls_after_scroll = list(
    dict.fromkeys(
        book_urls_after_scroll
    )
)

print(
    "Unique book links after scroll:",
    len(unique_book_urls_after_scroll)
)

Unique book links after scroll: 31


## 39. Bookshop.org Collection Structure

### Results

The selected leadership and management list contained:

- 30 unique book-product links with visible anchor text
- 31 unique book-product URLs when all product hyperlinks were considered
- Initial page height: 4,557 pixels
- Page height after scrolling: 4,557 pixels
- No evidence of additional records being loaded through infinite scrolling

The difference between the 30 text-based product links and 31 URL-based product links is likely caused by a product hyperlink without visible anchor text, such as an image-based link.

### Interpretation

The selected Bookshop.org list is a relatively small curated collection rather than a large paginated catalogue.

Therefore, a single list cannot satisfy the project's target of approximately 1,000 scraped book records.

The website may still be suitable if multiple relevant leadership, management, business, strategy, communication, organizational behavior, and related collections can be systematically identified and combined.

Duplicate books appearing across collections will be retained in the raw collection where appropriate and resolved during the later data-cleaning and integration stages.

Before scaling the collection process, an individual product page will be inspected to determine the richness and usefulness of its available metadata.

In [43]:
test_book = book_links_df[
    book_links_df["text"]
    .str.contains(
        "The Five Dysfunctions of a Team",
        case=False,
        na=False
    )
].iloc[0]

test_book_url = test_book["href"]

print("Test book:")
print(test_book["text"])

print("\nProduct URL:")
print(test_book_url)

Test book:
The Five Dysfunctions of a Team

Product URL:
https://bookshop.org/p/books/the-five-dysfunctions-of-a-team-a-leadership-fable-20th-anniversary-edition-patrick-m-lencioni/70dcd7fcaddb388d?ean=9780787960759&listref=best-books-on-leadership-and-management&aid=565&bkshp-astro=t


In [44]:
driver.get(test_book_url)

time.sleep(5)

print("Page title:")
print(driver.title)

print("\nCurrent URL:")
print(driver.current_url)

print("\nPage source length:")
print(len(driver.page_source))

Page title:
Just a moment...

Current URL:
https://bookshop.org/p/books/the-five-dysfunctions-of-a-team-a-leadership-fable-20th-anniversary-edition-patrick-m-lencioni/70dcd7fcaddb388d?ean=9780787960759&listref=best-books-on-leadership-and-management&aid=565&bkshp-astro=t

Page source length:
29676


In [45]:
product_page_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(
    product_page_text[:6000]
)

bookshop.org
Performing security verification
This website uses a security service to protect against malicious bots. This page is displayed while the website verifies you are not a bot.
Ray ID: a3f2398e6a20af15
Performance and Security by Cloudflare
Privacy


In [46]:
metadata_terms = [
    "ISBN",
    "Publisher",
    "Publication",
    "Published",
    "Pages",
    "Language",
    "Format",
    "Description"
]

for term in metadata_terms:

    print(
        term,
        "->",
        term.lower() in product_page_text.lower()
    )

ISBN -> False
Publisher -> False
Publication -> False
Published -> False
Pages -> False
Language -> False
Format -> False
Description -> False


In [47]:
for tag in [
    "h1",
    "h2",
    "h3",
    "h4"
]:

    elements = driver.find_elements(
        By.TAG_NAME,
        tag
    )

    print(
        f"\n--- {tag.upper()} ---"
    )

    for element in elements:

        text = element.text.strip()

        if text:
            print(text)


--- H1 ---
bookshop.org

--- H2 ---
Performing security verification

--- H3 ---

--- H4 ---


In [48]:
images = driver.find_elements(
    By.TAG_NAME,
    "img"
)

print(
    "Images found:",
    len(images)
)

Images found: 1


In [49]:
image_records = []

for image in images:

    try:

        src = image.get_attribute(
            "src"
        )

        alt = image.get_attribute(
            "alt"
        )

        if src:

            image_records.append({
                "alt": alt,
                "src": src
            })

    except Exception:
        pass

images_df = pd.DataFrame(
    image_records
)

images_df.head(20)

,alt,src
0,Icon for bookshop.org,https://bookshop.org/favicon.ico


In [50]:
json_ld_elements = driver.find_elements(
    By.XPATH,
    '//script[@type="application/ld+json"]'
)

print(
    "JSON-LD blocks found:",
    len(json_ld_elements)
)

JSON-LD blocks found: 0


In [51]:
for i, element in enumerate(
    json_ld_elements
):

    content = element.get_attribute(
        "innerHTML"
    )

    print(
        f"\n--- JSON-LD BLOCK {i + 1} ---"
    )

    print(
        content[:4000]
    )

## 46. Bookshop.org Product-Page Access Limitation

### Product-Page Test

An individual product page was tested using the same standard Selenium browser session that successfully rendered the public leadership and management list.

The selected test record was:

**The Five Dysfunctions of a Team**

The product URL contained the EAN:

`9780787960759`

### Result

Instead of rendering the book product page, the website displayed a Cloudflare security-verification page.

Observed indicators included:

- Page title: `Just a moment...`
- Message: `Performing security verification`
- Notification that the website uses a security service to protect against malicious bots
- No accessible book-level bibliographic metadata
- No ISBN, publisher, publication date, page count, language, format, or description in the rendered verification page

### Interpretation

Bookshop.org's public collection page can be rendered using a standard Selenium-controlled browser.

However, navigation to individual product pages triggers an automated security-verification mechanism.

The project will not attempt to bypass or circumvent this security control.

### Decision

Bookshop.org may still be used for metadata available directly from publicly rendered collection pages.

Individual product pages will not be automatically scraped.

Additional bibliographic fields may later be obtained from the API dataset or another openly accessible source during data integration.

## 47. EAN Availability in Collection-Page URLs

### Purpose

The Bookshop.org product URLs observed on the collection page may contain an `ean` query parameter.

For books, a 13-digit EAN beginning with the standard book prefixes may correspond to an ISBN-13.

The URL parameters will therefore be extracted systematically without navigating to protected individual product pages.

The raw value will initially be stored as `ean`. Conversion or interpretation as ISBN-13 will be validated during the data-cleaning stage.

In [52]:
from urllib.parse import urlparse, parse_qs

def extract_ean(url):

    try:
        query_parameters = parse_qs(
            urlparse(url).query
        )

        return query_parameters.get(
            "ean",
            [None]
        )[0]

    except Exception:
        return None

In [53]:
book_links_df["ean"] = (
    book_links_df["href"]
    .apply(extract_ean)
)

book_links_df[
    [
        "text",
        "ean",
        "href"
    ]
].head(10)

,text,ean,href
0,The Book on Leadership,9780785288381,https://bookshop.org/p/books/the-book-on-leade...
1,It Worked for Me,9780062135131,https://bookshop.org/p/books/it-worked-for-me-...
2,Leadership,9781476795935,https://bookshop.org/p/books/leadership-in-tur...
3,The Five Dysfunctions of a Team,9780787960759,https://bookshop.org/p/books/the-five-dysfunct...
4,Leadership 2.0,9780974320694,https://bookshop.org/p/books/leadership-2-0-dr...
5,Culturally Responsive School Leadership,9781682532072,https://bookshop.org/p/books/culturally-respon...
6,Leadership From Below,9780578888125,https://bookshop.org/p/books/leadership-from-b...
7,The Five Graces of Life and Leadership,9781119864042,https://bookshop.org/p/books/the-five-graces-o...
8,On Leadership,9780029113127,https://bookshop.org/p/books/on-leadership-joh...
9,The Heart of Business,9781647820381,https://bookshop.org/p/books/the-heart-of-busi...


In [54]:
print(
    "Book records:",
    len(book_links_df)
)

print(
    "EAN available:",
    book_links_df["ean"].notna().sum()
)

print(
    "EAN missing:",
    book_links_df["ean"].isna().sum()
)

print(
    "Unique EAN values:",
    book_links_df["ean"].nunique()
)

Book records: 30
EAN available: 30
EAN missing: 0
Unique EAN values: 30


In [55]:
book_links_df["ean_length"] = (
    book_links_df["ean"]
    .astype("string")
    .str.len()
)

book_links_df[
    "ean_length"
].value_counts(
    dropna=False
)

ean_length
13    30
Name: count, dtype: Int64

In [58]:
driver.get(bookshop_url)

time.sleep(5)

print(driver.title)

Just a moment...


In [59]:
print("Page title:")
print(driver.title)

print("\nCurrent URL:")
print(driver.current_url)

print("\nPage source length:")
print(len(driver.page_source))

print("\nBody text preview:")

current_body_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(current_body_text[:1000])

Page title:
Just a moment...

Current URL:
https://bookshop.org/lists/best-books-on-leadership-and-management

Page source length:
28917

Body text preview:
bookshop.org
Performing security verification
This website uses a security service to protect against malicious bots. This page is displayed while the website verifies you are not a bot.
Ray ID: a3f23e155e0fd7b4
Performance and Security by Cloudflare
Privacy


In [61]:
print(
    "Title found in body text:",
    "The Five Dysfunctions of a Team"
    in current_body_text
)

print(
    "Title found in page source:",
    "The Five Dysfunctions of a Team"
    in driver.page_source
)

Title found in body text: False
Title found in page source: False


In [62]:
current_product_links = driver.find_elements(
    By.CSS_SELECTOR,
    'a[href*="/p/books/"]'
)

print(
    "Current product links:",
    len(current_product_links)
)

Current product links: 0


In [63]:
for link in current_product_links[:10]:

    print(
        repr(link.text),
        "->",
        link.get_attribute("href")
    )

## 50. Final Bookshop.org Assessment

### Collection Results

The Bookshop.org leadership and management collection initially rendered successfully through a standard Selenium-controlled Chrome browser.

The collection produced:

- 30 book records with visible product titles and URLs
- 30 EAN identifiers
- 0 missing EAN values
- 30 unique EAN values
- All extracted EAN values contained 13 digits

### Security Verification

After navigating from the collection page to an individual product page, Bookshop.org presented a Cloudflare security-verification page.

Subsequent attempts to return to the original collection page within the same browser session also displayed the verification page.

The verification page contained no book-product links or catalogue metadata.

### Interpretation

The initial Selenium session demonstrated that useful structured information could be obtained from the public collection page. However, repeated automated navigation triggered the website's anti-bot security mechanism.

The project will not attempt to bypass or circumvent this security control.

### Decision

Bookshop.org will not be used for large-scale automated collection.

The successfully collected records may be retained as evidence of the source-evaluation process, but they will not be treated as the project's primary web-scraped dataset.

A different publicly accessible source will be selected for scalable and reproducible web scraping.

In [64]:
bookshop_evaluation_path = (
    raw_data_path
    / "bookshop_collection_evaluation.csv"
)

book_links_df.to_csv(
    bookshop_evaluation_path,
    index=False
)

print(
    "Saved:",
    bookshop_evaluation_path
)

print(
    "Shape:",
    book_links_df.shape
)

Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/raw/bookshop_collection_evaluation.csv
Shape: (30, 4)


## 51. Primary Web Scraping Source Selection

Following the source-feasibility assessment, the project will evaluate a public Koha-based library catalogue as the primary source for large-scale web scraping.

### Rationale

The selected source should provide:

- strong leadership and management subject coverage
- publicly accessible HTML pages
- ISBN identifiers
- author information
- publication metadata
- subject classifications
- sufficient records for scalable collection
- reproducible access without bypassing security controls

Public Koha catalogues are particularly suitable because bibliographic records are presented in structured HTML and may contain title, author, publisher, publication year, ISBN, language, subjects, description, and classification metadata.

This source also provides methodological independence from the Open Library API dataset collected in Notebook 01.

### Collection Principle

Only publicly accessible catalogue pages will be collected.

No authentication bypass, CAPTCHA bypass, stealth browser configuration, proxy rotation, or other access-control circumvention will be used.

The raw scraped data will be preserved without overwriting or artificially filling missing values.

In [65]:
kit_url = "https://katalog.bibliothek.kit.edu/"

kit_response = requests.get(
    kit_url,
    headers=headers,
    timeout=30
)

print(
    "Status code:",
    kit_response.status_code
)

print(
    "Response length:",
    len(kit_response.text)
)

print(
    "Content type:",
    kit_response.headers.get(
        "Content-Type"
    )
)

Status code: 200
Response length: 46417
Content type: text/html; charset=UTF-8


In [66]:
kit_soup = BeautifulSoup(
    kit_response.text,
    "html.parser"
)

print(
    "Page title:",
    kit_soup.title.get_text(
        " ",
        strip=True
    )
    if kit_soup.title
    else None
)

kit_text = kit_soup.get_text(
    " ",
    strip=True
)

print(
    "Page text length:",
    len(kit_text)
)

print(
    kit_text[:2000]
)

Page title: Koha online catalog
Page text length: 2136
Koha online catalog Skip to main content Koha online Your cart is empty. Cart Lists Your lists Log in to create your own lists Languages Deutsch English Log in to your account Your cookies Search history Search the catalog by: Library catalog Title Author Subject ISBN ISSN Series Call number Search the catalog by keyword Advanced search Course reserves Libraries Articles and Papers | Electronic Journals | Databases | Interlibrary Loan | KITopen media | Purchase request | Catalog Guide Log in to your account Log in with Hochschulaccount If you do not have an external account, but do have a local account, you can still log in: Card number: Password: Don't have a KIT library account yet? More Information on Registration / Opening of library account Forgot your password? Home 🕑 Opening hours at Library Campus South: Mon – Sun: Open 24 hours a day Service hours: Mon – Fri 9 am – 6 pm Access outside of service hours and on public holiday

In [67]:
search_forms = kit_soup.find_all("form")

print(
    "Number of forms:",
    len(search_forms)
)

for i, form in enumerate(search_forms):
    print(
        f"\n--- FORM {i + 1} ---"
    )
    print(
        "Action:",
        form.get("action")
    )
    print(
        "Method:",
        form.get("method")
    )

    inputs = form.find_all(
        ["input", "select"]
    )

    for item in inputs:
        print(
            item.name,
            "| name:",
            item.get("name"),
            "| value:",
            item.get("value")
        )

Number of forms: 3

--- FORM 1 ---
Action: /cgi-bin/koha/opac-search.pl
Method: get
select | name: idx | value: None
input | name: q | value: 
input | name: weight_search | value: 1

--- FORM 2 ---
Action: /cgi-bin/koha/opac-user.pl
Method: post
input | name: csrf_token | value: e3111441ee863fe944a2bc900b0d6cf0055f875f,e256c6bd11308c101ccda3ea23217d25d9836a1c,1790090131
input | name: has-search-query | value: 
input | name: koha_login_context | value: opac
input | name: login_userid | value: None
input | name: login_password | value: None
input | name: op | value: cud-login
input | name: None | value: Log in

--- FORM 3 ---
Action: /cgi-bin/koha/opac-user.pl
Method: post
input | name: csrf_token | value: e3111441ee863fe944a2bc900b0d6cf0055f875f,e256c6bd11308c101ccda3ea23217d25d9836a1c,1790090131
input | name: koha_login_context | value: opac
input | name: login_userid | value: None
input | name: login_password | value: None
input | name: op | value: cud-login
input | name: None | value

In [68]:
search_links = []

for link in kit_soup.find_all(
    "a",
    href=True
):

    href = link.get("href")
    text = link.get_text(
        " ",
        strip=True
    )

    if (
        "search" in href.lower()
        or "search" in text.lower()
    ):
        search_links.append({
            "text": text,
            "href": href
        })

search_links_df = pd.DataFrame(
    search_links
)

search_links_df

,text,href
0,Search history,/cgi-bin/koha/opac-search-history.pl
1,Advanced search,/cgi-bin/koha/opac-search.pl
2,Articles and Papers,https://katalog.bibliothek.kit.edu/plugin/Koha...
3,Electronic Journals,http://ezb.uni-regensburg.de/ezeit/search.phtm...
4,Searching in Articles & Papers,https://katalog.bibliothek.kit.edu/plugin/Koha...
5,Searching in KVK,https://kvk.bibliothek.kit.edu/?kataloge=K10PL...
6,Searching in Karlsruhe Library Portal,https://www.bibliotheksportal-karlsruhe.de/?la...


In [69]:
kit_leadership_url = (
    "https://katalog.bibliothek.kit.edu/"
    "cgi-bin/koha/opac-search.pl"
    "?q=leadership"
)

leadership_response = requests.get(
    kit_leadership_url,
    headers=headers,
    timeout=30
)

print(
    "Status code:",
    leadership_response.status_code
)

print(
    "Final URL:",
    leadership_response.url
)

print(
    "Response length:",
    len(leadership_response.text)
)

print(
    "Content type:",
    leadership_response.headers.get(
        "Content-Type"
    )
)

Status code: 200
Final URL: https://katalog.bibliothek.kit.edu/cgi-bin/koha/opac-search.pl?q=leadership
Response length: 315
Content type: text/html


In [70]:
leadership_soup = BeautifulSoup(
    leadership_response.text,
    "html.parser"
)

print(
    "Page title:",
    leadership_soup.title.get_text(
        " ",
        strip=True
    )
    if leadership_soup.title
    else None
)

leadership_text = leadership_soup.get_text(
    " ",
    strip=True
)

print(
    "\nText length:",
    len(leadership_text)
)

print(
    "\nFirst 3000 characters:"
)

print(
    leadership_text[:3000]
)

Page title: None

Text length: 0

First 3000 characters:



In [71]:
print(
    leadership_response.text
)

<html>
    <head>
        <link href="/opac-tmpl/lib/koha_fast_challenge/style.css" rel="stylesheet"/>
        <script src="/opac-tmpl/lib/koha_fast_challenge/index.js"></script>
    </head>
    <body>
        <div class="loading-overlay">
            <div class="spinner"></div>
        </div>
    </body>
</html>



In [72]:
print(
    repr(
        leadership_response.text
    )
)

'<html>\n    <head>\n        <link href="/opac-tmpl/lib/koha_fast_challenge/style.css" rel="stylesheet"/>\n        <script src="/opac-tmpl/lib/koha_fast_challenge/index.js"></script>\n    </head>\n    <body>\n        <div class="loading-overlay">\n            <div class="spinner"></div>\n        </div>\n    </body>\n</html>\n'


In [73]:
print(
    "Response headers:"
)

for key, value in leadership_response.headers.items():
    print(
        key,
        ":",
        value
    )

Response headers:
Date : Tue, 22 Sep 2026 15:16:14 GMT
Server : Apache
Strict-Transport-Security : max-age=63072000
Upgrade : h2,h2c
Connection : Upgrade, Keep-Alive
Last-Modified : Wed, 29 Oct 2025 19:02:32 GMT
Accept-Ranges : bytes
Vary : Accept-Encoding,User-Agent
Content-Encoding : gzip
Content-Length : 180
Keep-Alive : timeout=2, max=100
Content-Type : text/html


In [74]:
print(
    "Redirect history:",
    leadership_response.history
)

print(
    "Number of redirects:",
    len(leadership_response.history)
)

for response in leadership_response.history:
    print(
        response.status_code,
        "->",
        response.url
    )

Redirect history: []
Number of redirects: 0


In [75]:
print(
    "Contains JavaScript:",
    "<script" in leadership_response.text.lower()
)

print(
    "Contains meta refresh:",
    "http-equiv" in leadership_response.text.lower()
)

print(
    "Contains iframe:",
    "<iframe" in leadership_response.text.lower()
)

Contains JavaScript: True
Contains meta refresh: False
Contains iframe: False


In [76]:
search_forms = kit_soup.find_all(
    "form"
)

print(
    "Number of forms:",
    len(search_forms)
)

for i, form in enumerate(
    search_forms
):

    print(
        f"\n--- FORM {i + 1} ---"
    )

    print(
        "Action:",
        form.get("action")
    )

    print(
        "Method:",
        form.get("method")
    )

    inputs = form.find_all(
        [
            "input",
            "select"
        ]
    )

    for item in inputs:

        print(
            item.name,
            "| name:",
            item.get("name"),
            "| value:",
            item.get("value")
        )

Number of forms: 3

--- FORM 1 ---
Action: /cgi-bin/koha/opac-search.pl
Method: get
select | name: idx | value: None
input | name: q | value: 
input | name: weight_search | value: 1

--- FORM 2 ---
Action: /cgi-bin/koha/opac-user.pl
Method: post
input | name: csrf_token | value: e3111441ee863fe944a2bc900b0d6cf0055f875f,e256c6bd11308c101ccda3ea23217d25d9836a1c,1790090131
input | name: has-search-query | value: 
input | name: koha_login_context | value: opac
input | name: login_userid | value: None
input | name: login_password | value: None
input | name: op | value: cud-login
input | name: None | value: Log in

--- FORM 3 ---
Action: /cgi-bin/koha/opac-user.pl
Method: post
input | name: csrf_token | value: e3111441ee863fe944a2bc900b0d6cf0055f875f,e256c6bd11308c101ccda3ea23217d25d9836a1c,1790090131
input | name: koha_login_context | value: opac
input | name: login_userid | value: None
input | name: login_password | value: None
input | name: op | value: cud-login
input | name: None | value

## 59. KIT Library Catalogue Feasibility Assessment

### Search Architecture

The KIT Library public catalogue uses a Koha-based search interface.

Inspection of the public search form identified the following search configuration:

- Endpoint: `/cgi-bin/koha/opac-search.pl`
- Method: `GET`
- Search-field parameter: `idx`
- Query parameter: `q`
- Search-weight parameter: `weight_search`

The catalogue homepage was successfully retrieved through a standard HTTP request and returned the public search interface.

### Search Endpoint Test

A test search for `leadership` returned HTTP status code `200`.

However, the response contained only a small JavaScript challenge page rather than catalogue search results.

The returned HTML referenced:

- `koha_fast_challenge/style.css`
- `koha_fast_challenge/index.js`

and displayed a loading overlay.

There was no HTTP redirect, and no bibliographic search-result content was present in the returned HTML.

### Interpretation

Although the KIT catalogue is publicly searchable through a normal web browser, its search endpoint implements a JavaScript-based challenge for automated requests.

A successful HTTP status code therefore does not indicate successful retrieval of catalogue records.

### Decision

The project will not attempt to reverse-engineer or bypass the catalogue's automated-request challenge.

KIT Library will therefore not be used as the primary large-scale scraping source.

This decision supports the project's requirements for:

- reproducibility
- transparent data collection
- responsible web scraping
- avoidance of access-control circumvention

## 60. Revised Primary Scraping Candidate — LeadershipNow / LeaderShop

Following the accessibility limitations identified with several commercial bookstores and library catalogues, the project identified LeadershipNow / LeaderShop as a stronger domain-specific candidate.

### Why This Source Is Relevant

LeadershipNow focuses specifically on leadership and management-related books.

Public book-list pages expose bibliographic and product metadata including:

- book title
- author
- ISBN
- publisher
- publication date
- format
- page count
- cover image
- link to additional book information

The source is therefore highly aligned with the project's leadership and management scope.

### Methodological Advantage

Unlike a general bookstore catalogue, the source already provides a degree of domain filtering because its collections focus on leadership, management, organizational development, strategy, communication, and related professional topics.

The source will first be evaluated for:

1. HTTP accessibility
2. HTML structure
3. number of available book records
4. pagination or archive structure
5. metadata consistency
6. scalability

Only after these checks will large-scale collection begin.

In [78]:
leadershipnow_url = (
    "https://www.leadershipnow.com/"
    "leadershop/new.html"
)

leadershipnow_response = requests.get(
    leadershipnow_url,
    headers=headers,
    timeout=30
)

print(
    "Status code:",
    leadershipnow_response.status_code
)

print(
    "Final URL:",
    leadershipnow_response.url
)

print(
    "Response length:",
    len(leadershipnow_response.text)
)

print(
    "Content type:",
    leadershipnow_response.headers.get(
        "Content-Type"
    )
)

Status code: 200
Final URL: https://www.leadershipnow.com/leadershop/new.html
Response length: 247271
Content type: text/html


In [79]:
leadershipnow_soup = BeautifulSoup(
    leadershipnow_response.text,
    "html.parser"
)

print(
    "Page title:",
    leadershipnow_soup.title.get_text(
        " ",
        strip=True
    )
    if leadershipnow_soup.title
    else None
)

leadershipnow_text = (
    leadershipnow_soup.get_text(
        "\n",
        strip=True
    )
)

print(
    "Text length:",
    len(leadershipnow_text)
)

print(
    leadershipnow_text[:4000]
)

Page title: Leadership Books and Resources | New and Future Releases - LeaderShop @ LeadershipNow.com
Text length: 54605
Leadership Books and Resources | New and Future Releases - LeaderShop @ LeadershipNow.com
input.gsc-search-button { width:36px; }
Home
|
LeadingBlog
|
Podcast
|
LeadingThoughts
|
LeadershipMinute
|
LeadingArticles
|
LeaderShop
|
Contact
|
About
In this section, you will find the top new and future releases and some of the more noteworthy current releases on the issues facing leaders.
January

February

March

April

May

June

July

August

September

October

November

December
2025 Releases
|
2024 Releases
|
2023 Releases
|
2022 Releases
|
2021 Releases
|
2020 Releases
|
2019 Releases
|
Previous Years
Best Leadership Books of 2025
|   Title Index:
A-K
|   Title Index:
L-Z
|
Search Books
|
LeaderShop Main Page
Note: The Amazon links below are affiliate links to books. If you click through and purchase, we will receive a small commission on the sale.
These

In [80]:
metadata_terms = [
    "ISBN:",
    "Publisher:",
    "Pub. Date:",
    "Format:"
]

for term in metadata_terms:
    print(
        term,
        "->",
        term in leadershipnow_text
    )

ISBN: -> True
Publisher: -> True
Pub. Date: -> True
Format: -> True


In [81]:
print(
    "ISBN occurrences:",
    leadershipnow_text.count(
        "ISBN:"
    )
)

print(
    "Publisher occurrences:",
    leadershipnow_text.count(
        "Publisher:"
    )
)

print(
    "Publication date occurrences:",
    leadershipnow_text.count(
        "Pub. Date:"
    )
)

ISBN occurrences: 227
Publisher occurrences: 227
Publication date occurrences: 227


## 63. LeadershipNow Initial Feasibility Results

The LeadershipNow / LeaderShop new and future releases page was successfully retrieved using a standard HTTP request.

### Accessibility Results

- HTTP status: 200
- Response size: 247,271 characters
- Parsed text size: 54,605 characters
- No authentication or browser automation was required

### Metadata Coverage

The page contained 227 book records based on repeated bibliographic metadata fields.

Observed metadata occurrences:

- ISBN: 227
- Publisher: 227
- Publication date: 227
- Format: 227

The page also exposes:

- book title
- author
- page count
- ISBN
- publisher
- publication date
- links to additional book information

### Interpretation

LeadershipNow provides substantially better domain relevance and metadata consistency than the previously evaluated sources.

The page is specifically curated around leadership and related management topics, reducing the amount of irrelevant material that would otherwise require filtering.

The consistent repetition of ISBN, publisher, publication date, and format fields indicates that the page has a sufficiently regular structure for systematic extraction.

### Preliminary Decision

LeadershipNow will proceed to the scalability and HTML-structure evaluation stage.

Before large-scale scraping begins, the project will determine:

1. the number of historical release pages available;
2. the approximate number of records available across those pages;
3. whether records are duplicated between yearly pages;
4. whether the HTML structure remains consistent across years; and
5. whether approximately 1,000 relevant raw records can be collected reproducibly.

In [82]:
from urllib.parse import urljoin

leadershipnow_links = []

for link in leadershipnow_soup.find_all(
    "a",
    href=True
):
    link_text = link.get_text(
        " ",
        strip=True
    )

    link_url = urljoin(
        leadershipnow_url,
        link.get("href")
    )

    leadershipnow_links.append({
        "text": link_text,
        "url": link_url
    })

leadershipnow_links_df = pd.DataFrame(
    leadershipnow_links
)

print(
    "Total links:",
    len(leadershipnow_links_df)
)

leadershipnow_links_df.head(20)

Total links: 735


,text,url
0,,https://www.leadershipnow.com/index.html
1,Home,https://www.leadershipnow.com/index.html
2,LeadingBlog,https://www.leadershipnow.com/leadingblog/inde...
3,Podcast,https://www.leadershipnow.com/podcast/index.html
4,LeadingThoughts,https://www.leadershipnow.com/quotes.html
5,LeadershipMinute,https://www.leadershipnow.com/minute.html
6,LeadingArticles,https://www.leadershipnow.com/articles.html
7,LeaderShop,https://www.leadershipnow.com/leadershop/index...
8,Contact,https://www.leadershipnow.com/contact.html
9,About,https://www.leadershipnow.com/about.html


In [83]:
year_links_df = (
    leadershipnow_links_df[
        leadershipnow_links_df["text"]
        .str.contains(
            r"20\d{2}|Previous Years",
            regex=True,
            na=False
        )
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

year_links_df

,text,url
0,2025 Releases,https://www.leadershipnow.com/leadershop/new20...
1,2024 Releases,https://www.leadershipnow.com/leadershop/new20...
2,2023 Releases,https://www.leadershipnow.com/leadershop/new20...
3,2022 Releases,https://www.leadershipnow.com/leadershop/new20...
4,2021 Releases,https://www.leadershipnow.com/leadershop/new20...
5,2020 Releases,https://www.leadershipnow.com/leadershop/new20...
6,2019 Releases,https://www.leadershipnow.com/leadershop/new20...
7,Previous Years,https://www.leadershipnow.com/leadershop/new20...
8,Best Leadership Books of 2025,https://www.leadershipnow.com/leadershop/best2...


In [84]:
leadershop_links_df = (
    leadershipnow_links_df[
        leadershipnow_links_df["url"]
        .str.contains(
            "/leadershop/",
            case=False,
            na=False
        )
    ]
    .drop_duplicates(
        subset=["url"]
    )
    .reset_index(drop=True)
)

print(
    "Unique LeaderShop URLs:",
    len(leadershop_links_df)
)

leadershop_links_df[
    ["text", "url"]
].head(100)

Unique LeaderShop URLs: 29


,text,url
0,LeaderShop,https://www.leadershipnow.com/leadershop/index...
1,January,https://www.leadershipnow.com/leadershop/new.h...
2,February,https://www.leadershipnow.com/leadershop/new.h...
3,March,https://www.leadershipnow.com/leadershop/new.h...
4,April,https://www.leadershipnow.com/leadershop/new.h...
5,May,https://www.leadershipnow.com/leadershop/new.h...
6,June,https://www.leadershipnow.com/leadershop/new.h...
7,July,https://www.leadershipnow.com/leadershop/new.h...
8,August,https://www.leadershipnow.com/leadershop/new.h...
9,September,https://www.leadershipnow.com/leadershop/new.h...


In [85]:
test_isbn = "9781668066386"

isbn_text_node = leadershipnow_soup.find(
    string=lambda text:
        text
        and test_isbn in text
)

print(
    "ISBN node:",
    repr(isbn_text_node)
)

ISBN node: ' 9781668066386'


In [86]:
if isbn_text_node:

    parent = isbn_text_node.parent

    print(
        "Parent tag:",
        parent.name
    )

    print(
        "\nParent HTML:"
    )

    print(
        parent.prettify()[:5000]
    )

Parent tag: font

Parent HTML:
<font face="arial,helvetica,sans-serif" size="2">
 <b>
  Format:
 </b>
 Hardcover, 288 pages
 <br/>
 <b>
  ISBN:
 </b>
 9781668066386
 <br/>
 <b>
  Publisher:
 </b>
 Simon Element / Simon Acumen
 <br/>
 <b>
  Pub. Date:
 </b>
 September 1, 2026
 <br/>
 <br/>
 <a href="https://amzn.to/4mUClQu" target="_blank">
  <img align="absbottom" border="0" height="19" src="images/amazonbutton2.gif" width="82"/>
 </a>
 <img border="0" height="8" src="images/orange_arrow.gif" width="7"/>
 <a href="https://amzn.to/4mUClQu" target="_blank">
  More Information on this Book
 </a>
</font>



In [87]:
if isbn_text_node:

    current = isbn_text_node.parent

    for level in range(1, 6):

        current = current.parent

        if current is None:
            break

        print(
            f"\n--- ANCESTOR LEVEL {level} ---"
        )

        print(
            "Tag:",
            current.name
        )

        print(
            "Class:",
            current.get("class")
        )

        print(
            current.prettify()[:4000]
        )


--- ANCESTOR LEVEL 1 ---
Tag: font
Class: None
<font color="#000000" face="georgia, palantino, times new roman, serif" size="4">
 <a name="September">
 </a>
 <br/>
 <a href="#top">
  <img alt="go to top" border="0" height="21" src="https://www.leadershipnow.com/images/top.GIF" vspace="3" width="48"/>
 </a>
 <br/>
 <br/>
 <table border="0" cellpadding="2" cellspacing="0" width="1160">
  <tr>
   <td align="left" bgcolor="#f5f2ee">
    <font face="georgia, verdana" size="4">
     September 2026
    </font>
   </td>
   <td align="right" bgcolor="#f5f2ee">
    <img alt="leadershop" border="0" height="15" src="images/anglearrow.gif" width="15"/>
   </td>
  </tr>
 </table>
 <br/>
 <img alt="leadership books" border="0" height="1" src="images/ruledot.gif" vspace="3" width="1160"/>
 <br/>
 <a href="https://amzn.to/4mUClQu" target="_blank">
  <img align="left" alt="9781668066386" border="0" height="200" hspace="15" src="https://www.leadershipnow.com/leadershop/images/9781668066386.jpg" vspace="

In [88]:
for _, row in year_links_df.iterrows():
    print(
        row["text"],
        "->",
        row["url"]
    )

2025 Releases -> https://www.leadershipnow.com/leadershop/new2025.html
2024 Releases -> https://www.leadershipnow.com/leadershop/new2024.html
2023 Releases -> https://www.leadershipnow.com/leadershop/new2023.html
2022 Releases -> https://www.leadershipnow.com/leadershop/new2022.html
2021 Releases -> https://www.leadershipnow.com/leadershop/new2021.html
2020 Releases -> https://www.leadershipnow.com/leadershop/new2020.html
2019 Releases -> https://www.leadershipnow.com/leadershop/new2019.html
Previous Years -> https://www.leadershipnow.com/leadershop/new2017.html
Best Leadership Books of 2025 -> https://www.leadershipnow.com/leadershop/best2025.html


In [89]:
archive_test_results = []

for _, row in year_links_df.iterrows():

    page_name = row["text"]
    page_url = row["url"]

    try:

        response = requests.get(
            page_url,
            headers=headers,
            timeout=30
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        page_text = soup.get_text(
            "\n",
            strip=True
        )

        archive_test_results.append({
            "page": page_name,
            "url": page_url,
            "status_code": response.status_code,
            "response_length": len(response.text),
            "isbn_count": page_text.count("ISBN:"),
            "publisher_count": page_text.count("Publisher:"),
            "publication_date_count": page_text.count("Pub. Date:")
        })

        print(
            page_name,
            "| Status:",
            response.status_code,
            "| ISBN:",
            page_text.count("ISBN:")
        )

        time.sleep(1)

    except Exception as error:

        archive_test_results.append({
            "page": page_name,
            "url": page_url,
            "status_code": None,
            "response_length": None,
            "isbn_count": None,
            "publisher_count": None,
            "publication_date_count": None,
            "error": str(error)
        })

        print(
            page_name,
            "| ERROR:",
            error
        )

2025 Releases | Status: 200 | ISBN: 252
2024 Releases | Status: 200 | ISBN: 280
2023 Releases | Status: 200 | ISBN: 303
2022 Releases | Status: 200 | ISBN: 289
2021 Releases | Status: 200 | ISBN: 294
2020 Releases | Status: 200 | ISBN: 261
2019 Releases | Status: 200 | ISBN: 244
Previous Years | Status: 200 | ISBN: 137
Best Leadership Books of 2025 | Status: 200 | ISBN: 0


In [90]:
archive_test_df = pd.DataFrame(
    archive_test_results
)

archive_test_df

,page,url,status_code,response_length,isbn_count,publisher_count,publication_date_count
0,2025 Releases,https://www.leadershipnow.com/leadershop/new20...,200,266723,252,252,252
1,2024 Releases,https://www.leadershipnow.com/leadershop/new20...,200,294840,280,280,280
2,2023 Releases,https://www.leadershipnow.com/leadershop/new20...,200,317385,303,303,303
3,2022 Releases,https://www.leadershipnow.com/leadershop/new20...,200,302698,289,289,289
4,2021 Releases,https://www.leadershipnow.com/leadershop/new20...,200,302220,294,294,294
5,2020 Releases,https://www.leadershipnow.com/leadershop/new20...,200,274362,261,261,261
6,2019 Releases,https://www.leadershipnow.com/leadershop/new20...,200,253513,244,244,244
7,Previous Years,https://www.leadershipnow.com/leadershop/new20...,200,141621,137,137,137
8,Best Leadership Books of 2025,https://www.leadershipnow.com/leadershop/best2...,200,52088,0,0,0


In [91]:
print(
    "Pages tested:",
    len(archive_test_df)
)

print(
    "Successful pages:",
    (
        archive_test_df["status_code"]
        == 200
    ).sum()
)

print(
    "Total ISBN occurrences:",
    archive_test_df[
        "isbn_count"
    ].fillna(0).sum()
)

Pages tested: 9
Successful pages: 9
Total ISBN occurrences: 2060


In [92]:
archive_test_df[
    [
        "page",
        "status_code",
        "isbn_count",
        "publisher_count",
        "publication_date_count"
    ]
]

,page,status_code,isbn_count,publisher_count,publication_date_count
0,2025 Releases,200,252,252,252
1,2024 Releases,200,280,280,280
2,2023 Releases,200,303,303,303
3,2022 Releases,200,289,289,289
4,2021 Releases,200,294,294,294
5,2020 Releases,200,261,261,261
6,2019 Releases,200,244,244,244
7,Previous Years,200,137,137,137
8,Best Leadership Books of 2025,200,0,0,0


In [93]:
archive_test_df[
    "metadata_counts_match"
] = (
    (
        archive_test_df["isbn_count"]
        == archive_test_df["publisher_count"]
    )
    &
    (
        archive_test_df["isbn_count"]
        == archive_test_df["publication_date_count"]
    )
)

archive_test_df[
    [
        "page",
        "isbn_count",
        "publisher_count",
        "publication_date_count",
        "metadata_counts_match"
    ]
]

,page,isbn_count,publisher_count,publication_date_count,metadata_counts_match
0,2025 Releases,252,252,252,True
1,2024 Releases,280,280,280,True
2,2023 Releases,303,303,303,True
3,2022 Releases,289,289,289,True
4,2021 Releases,294,294,294,True
5,2020 Releases,261,261,261,True
6,2019 Releases,244,244,244,True
7,Previous Years,137,137,137,True
8,Best Leadership Books of 2025,0,0,0,True


## 71. Final Primary Web Scraping Source Selection

LeadershipNow / LeaderShop was selected as the primary web-scraping source for the project.

### Scalability Test

Historical release pages were tested before large-scale extraction.

All tested historical release pages returned HTTP status code 200.

The archive contained:

| Release Page | ISBN Records |
|---|---:|
| 2025 | 252 |
| 2024 | 280 |
| 2023 | 303 |
| 2022 | 289 |
| 2021 | 294 |
| 2020 | 261 |
| 2019 | 244 |
| Previous Years | 137 |

These historical pages contained 2,060 ISBN-bearing record occurrences.

The current 2026 release page contained an additional 227 records, producing an identified collection pool of approximately 2,287 raw record occurrences before deduplication.

### Metadata Consistency

For every historical release page tested, the number of:

- ISBN fields
- publisher fields
- publication-date fields

was identical.

This provides strong evidence that the historical pages follow a consistent bibliographic structure suitable for systematic extraction.

### Source Selection Decision

LeadershipNow / LeaderShop satisfies the project's primary scraping requirements:

- strong leadership and management relevance
- public accessibility
- standard HTTP retrieval
- sufficient collection scale
- structured bibliographic metadata
- ISBN identifiers
- publisher information
- publication dates
- format and page information
- cover images
- reproducible collection using Requests and BeautifulSoup

The project will therefore use LeadershipNow / LeaderShop as the primary web-scraping source.

Approximately 1,000 raw book records will be collected for the project.

The complete available archive does not need to be scraped because the project requirement is approximately 1,000 web-scraped records.

In [94]:
target_pages_df = (
    archive_test_df[
        archive_test_df["page"].isin([
            "2025 Releases",
            "2024 Releases",
            "2023 Releases",
            "2022 Releases"
        ])
    ]
    .copy()
    .reset_index(drop=True)
)

target_pages_df[
    [
        "page",
        "url",
        "isbn_count"
    ]
]

,page,url,isbn_count
0,2025 Releases,https://www.leadershipnow.com/leadershop/new20...,252
1,2024 Releases,https://www.leadershipnow.com/leadershop/new20...,280
2,2023 Releases,https://www.leadershipnow.com/leadershop/new20...,303
3,2022 Releases,https://www.leadershipnow.com/leadershop/new20...,289


In [95]:
print(
    "Target pages:",
    len(target_pages_df)
)

print(
    "Expected raw records:",
    target_pages_df[
        "isbn_count"
    ].sum()
)

Target pages: 4
Expected raw records: 1124


## 74. Raw Web-Scraping Schema

The scraper will collect bibliographic information as presented on the source pages.

### Raw Fields

- `scrape_id`
- `title`
- `subtitle`
- `author`
- `format_raw`
- `isbn`
- `publisher`
- `publication_date_raw`
- `cover_url`
- `external_book_url`
- `release_page`
- `release_page_url`
- `source`
- `scraped_at`

### Data Preservation Principle

Notebook 02 performs data collection rather than data cleaning.

Therefore:

- ISBN values will initially remain as displayed by the source.
- publication dates will remain as raw strings.
- format and page information will remain combined in `format_raw`.
- missing values will not be imputed.
- titles and author names will not be standardized.
- duplicate books will not be removed from the raw dataset.
- suspicious or inconsistent metadata will be preserved for later validation.

Data cleaning, parsing, normalization, and deduplication will be performed in Notebook 03.

In [96]:
test_page_url = target_pages_df.loc[
    target_pages_df["page"]
    == "2025 Releases",
    "url"
].iloc[0]

test_response = requests.get(
    test_page_url,
    headers=headers,
    timeout=30
)

test_soup = BeautifulSoup(
    test_response.text,
    "html.parser"
)

print(
    "Status:",
    test_response.status_code
)

print(
    "ISBN occurrences:",
    test_soup.get_text(
        "\n",
        strip=True
    ).count("ISBN:")
)

Status: 200
ISBN occurrences: 252


In [97]:
candidate_covers = []

for img in test_soup.find_all(
    "img",
    src=True
):

    alt_value = (
        img.get("alt")
        or ""
    ).strip()

    src_value = img.get("src")

    if (
        alt_value.isdigit()
        and len(alt_value) in [10, 13]
    ):

        candidate_covers.append({
            "image_alt": alt_value,
            "image_src": urljoin(
                test_page_url,
                src_value
            )
        })

candidate_covers_df = pd.DataFrame(
    candidate_covers
)

print(
    "Candidate covers:",
    len(candidate_covers_df)
)

candidate_covers_df.head(10)

Candidate covers: 251


,image_alt,image_src
0,9798891386211,https://www.leadershipnow.com/leadershop/image...
1,9781529154801,https://www.leadershipnow.com/leadershop/image...
2,9781668204191,https://www.leadershipnow.com/leadershop/image...
3,9781637747841,https://www.leadershipnow.com/leadershop/image...
4,9781636987897,https://www.leadershipnow.com/leadershop/image...
5,9781394388868,https://www.leadershipnow.com/leadershop/image...
6,9781640955837,https://www.leadershipnow.com/leadershop/image...
7,9781394313709,https://www.leadershipnow.com/leadershop/image...
8,9798886454659,https://www.leadershipnow.com/leadershop/image...
9,9798216391746,https://www.leadershipnow.com/leadershop/image...


In [98]:
first_cover = None

for img in test_soup.find_all(
    "img",
    src=True
):

    alt_value = (
        img.get("alt")
        or ""
    ).strip()

    if (
        alt_value.isdigit()
        and len(alt_value) in [10, 13]
    ):
        first_cover = img
        break

print(
    "Cover alt:",
    first_cover.get("alt")
    if first_cover
    else None
)

print(
    "Cover src:",
    first_cover.get("src")
    if first_cover
    else None
)

Cover alt: 9798891386211
Cover src: https://www.leadershipnow.com/leadershop/images/9798891386211.jpg


In [99]:
if first_cover:

    parent_link = first_cover.find_parent(
        "a"
    )

    print(
        "Parent link:",
        parent_link.get("href")
        if parent_link
        else None
    )

    print(
        "\nParent link text:"
    )

    print(
        parent_link.get_text(
            " ",
            strip=True
        )
        if parent_link
        else None
    )

Parent link: https://amzn.to/3WhDMvU

Parent link text:
The Seismic Shift in You


In [100]:
if first_cover:

    container = first_cover.parent.parent

    print(
        container.prettify()[:5000]
    )

<font color="#000000" face="georgia, palantino, times new roman, serif" size="4">
 <a name="December">
 </a>
 <br/>
 <a href="#top">
  <img alt="go to top" border="0" height="21" src="https://www.leadershipnow.com/images/top.GIF" vspace="3" width="48"/>
 </a>
 <br/>
 <br/>
 <table border="0" cellpadding="2" cellspacing="0" width="1160">
  <tr>
   <td align="left" bgcolor="#f5f2ee">
    <font face="georgia, verdana" size="4">
     December 2025
    </font>
   </td>
   <td align="right" bgcolor="#f5f2ee">
    <img alt="leadershop" border="0" height="15" src="images/anglearrow.gif" width="15"/>
   </td>
  </tr>
 </table>
 <br/>
 <img alt="leadership books" border="0" height="1" src="images/ruledot.gif" vspace="3" width="1160"/>
 <br/>
 <a href="https://amzn.to/3WhDMvU" target="_blank">
  <img align="left" alt="9798891386211" border="0" height="200" hspace="15" src="https://www.leadershipnow.com/leadershop/images/9798891386211.jpg" vspace="5" width="133"/>
  The Seismic Shift in You
 </a>


## 78. LeadershipNow Record Structure

Inspection of the 2025 release-page HTML confirmed a consistent repeating bibliographic pattern.

A typical record contains:

1. cover image
2. book title
3. subtitle, when available
4. author
5. format and page information
6. ISBN
7. publisher
8. publication date
9. external book-information link

### Important Structural Finding

The 2025 page contains:

- 252 displayed ISBN records
- 251 cover images whose `alt` attribute resembles an ISBN

Therefore, cover images cannot be used as the authoritative record-counting mechanism.

The displayed `ISBN:` metadata field will be treated as the primary record anchor.

Cover-image ISBN information will only be used as a supplementary validation field.

This approach prevents books with missing or inconsistent cover metadata from being unintentionally excluded.

In [101]:
from bs4 import NavigableString

def get_text_after_label(label_tag):
    """
    Return the first meaningful text value appearing
    after a metadata label such as ISBN:, Publisher:,
    Format:, or Pub. Date:.
    """

    for element in label_tag.next_elements:

        if element is label_tag:
            continue

        if (
            getattr(element, "name", None)
            == "br"
        ):
            break

        if isinstance(
            element,
            NavigableString
        ):

            value = str(element).strip()

            if value:
                return value

    return None

In [102]:
first_isbn_label = test_soup.find(
    "b",
    string=lambda text:
        text
        and text.strip() == "ISBN:"
)

print(
    "Label:",
    first_isbn_label.get_text(
        strip=True
    )
)

print(
    "Value:",
    get_text_after_label(
        first_isbn_label
    )
)

Label: ISBN:
Value: ISBN:


In [103]:
first_metadata_block = (
    first_isbn_label.find_parent(
        "font"
    )
)

print(
    first_metadata_block.prettify()[
        :2500
    ]
)

<font face="arial,helvetica,sans-serif" size="2">
 <b>
  Format:
 </b>
 Hardcover, 152 pages
 <br/>
 <b>
  ISBN:
 </b>
 9798891386211
 <br/>
 <b>
  Publisher:
 </b>
 100 Coaches Publishing
 <br/>
 <b>
  Pub. Date:
 </b>
 December 2, 2025
 <br/>
 <br/>
 <a href="https://amzn.to/3WhDMvU" target="_blank">
  <img align="absbottom" border="0" height="19" src="images/amazonbutton2.gif" width="82"/>
 </a>
 <img border="0" height="8" src="images/orange_arrow.gif" width="7"/>
 <a href="https://amzn.to/3WhDMvU" target="_blank">
  More Information on this Book
 </a>
</font>



In [104]:
def extract_metadata_block(
    metadata_block
):

    metadata = {}

    for label in metadata_block.find_all(
        "b"
    ):

        label_name = label.get_text(
            " ",
            strip=True
        )

        value = get_text_after_label(
            label
        )

        metadata[
            label_name
        ] = value

    return metadata

In [105]:
first_metadata = (
    extract_metadata_block(
        first_metadata_block
    )
)

first_metadata

{'Format:': 'Format:',
 'ISBN:': 'ISBN:',
 'Publisher:': 'Publisher:',
 'Pub. Date:': 'Pub. Date:'}

In [107]:
previous_elements = []

for element in first_metadata_block.previous_elements:

    if isinstance(
        element,
        NavigableString
    ):

        value = str(
            element
        ).strip()

        if value:
            previous_elements.append(
                value
            )

    if len(
        previous_elements
    ) >= 10:
        break

previous_elements

['Michelle Johnston and Marshall Goldsmith',
 ': The Seven Necessary Shifts to Create Connection and Drive Results',
 'The Seismic Shift in You',
 'December 2025',
 'These links are provided for your convenience and importantly, help to support our work here. We appreciate your use of these links.',
 'Note: The Amazon links below are affiliate links to books. If you click through and purchase, we will receive a small commission on the sale.',
 'LeaderShop Main Page',
 '|',
 'Search Books',
 '|']

In [108]:
from bs4 import NavigableString

def get_text_after_label(label_tag):
    """
    Extract the first meaningful text value after a
    metadata label and before the next <br> tag.
    """

    for element in label_tag.next_siblings:

        if getattr(element, "name", None) == "br":
            break

        if isinstance(element, NavigableString):

            value = str(element).strip()

            if value:
                return value

    return None

In [109]:
first_metadata = extract_metadata_block(
    first_metadata_block
)

first_metadata

{'Format:': 'Hardcover,\xa0152 pages',
 'ISBN:': '9798891386211',
 'Publisher:': '100 Coaches Publishing',
 'Pub. Date:': 'December 2, 2025'}

In [110]:
def extract_book_identity(metadata_block):

    author_tag = metadata_block.find_previous(
        "i"
    )

    author = (
        author_tag.get_text(
            " ",
            strip=True
        )
        if author_tag
        else None
    )

    title_link = (
        author_tag.find_previous(
            "a"
        )
        if author_tag
        else None
    )

    title = (
        title_link.get_text(
            " ",
            strip=True
        )
        if title_link
        else None
    )

    return {
        "title": title,
        "author": author
    }

In [111]:
first_identity = extract_book_identity(
    first_metadata_block
)

first_identity

{'title': 'The Seismic Shift in You',
 'author': 'Michelle Johnston and Marshall Goldsmith'}

In [112]:
def extract_subtitle(
    title_link,
    author_tag
):

    subtitle_parts = []

    for element in title_link.next_siblings:

        if element == author_tag:
            break

        if getattr(element, "name", None) == "i":
            break

        if getattr(element, "name", None) == "br":
            break

        if isinstance(
            element,
            NavigableString
        ):

            value = str(
                element
            ).strip()

            if value:
                subtitle_parts.append(
                    value
                )

    subtitle = " ".join(
        subtitle_parts
    ).strip()

    return (
        subtitle
        if subtitle
        else None
    )

In [113]:
author_tag = (
    first_metadata_block.find_previous(
        "i"
    )
)

title_link = (
    author_tag.find_previous(
        "a"
    )
)

first_subtitle = extract_subtitle(
    title_link,
    author_tag
)

print(
    "Title:",
    title_link.get_text(
        " ",
        strip=True
    )
)

print(
    "Subtitle:",
    first_subtitle
)

print(
    "Author:",
    author_tag.get_text(
        " ",
        strip=True
    )
)

Title: The Seismic Shift in You
Subtitle: : The Seven Necessary Shifts to Create Connection and Drive Results
Author: Michelle Johnston and Marshall Goldsmith


In [114]:
def parse_book_record(
    isbn_label,
    page_name,
    page_url
):

    metadata_block = (
        isbn_label.find_parent(
            "font"
        )
    )

    if metadata_block is None:
        return None

    metadata = extract_metadata_block(
        metadata_block
    )

    author_tag = (
        metadata_block.find_previous(
            "i"
        )
    )

    title_link = (
        author_tag.find_previous(
            "a"
        )
        if author_tag
        else None
    )

    title = (
        title_link.get_text(
            " ",
            strip=True
        )
        if title_link
        else None
    )

    author = (
        author_tag.get_text(
            " ",
            strip=True
        )
        if author_tag
        else None
    )

    subtitle = (
        extract_subtitle(
            title_link,
            author_tag
        )
        if title_link
        and author_tag
        else None
    )

    external_book_url = (
        urljoin(
            page_url,
            title_link.get("href")
        )
        if title_link
        and title_link.get("href")
        else None
    )

    cover_img = (
        title_link.find(
            "img",
            src=True
        )
        if title_link
        else None
    )

    cover_url = (
        urljoin(
            page_url,
            cover_img.get("src")
        )
        if cover_img
        else None
    )

    cover_alt = (
        cover_img.get("alt")
        if cover_img
        else None
    )

    return {
        "title": title,
        "subtitle": subtitle,
        "author": author,
        "format_raw": metadata.get(
            "Format:"
        ),
        "isbn": metadata.get(
            "ISBN:"
        ),
        "publisher": metadata.get(
            "Publisher:"
        ),
        "publication_date_raw": metadata.get(
            "Pub. Date:"
        ),
        "cover_url": cover_url,
        "cover_alt_raw": cover_alt,
        "external_book_url": external_book_url,
        "release_page": page_name,
        "release_page_url": page_url,
        "source": "LeadershipNow / LeaderShop"
    }

In [115]:
first_record = parse_book_record(
    first_isbn_label,
    "2025 Releases",
    test_page_url
)

first_record

{'title': 'The Seismic Shift in You',
 'subtitle': ': The Seven Necessary Shifts to Create Connection and Drive Results',
 'author': 'Michelle Johnston and Marshall Goldsmith',
 'format_raw': 'Hardcover,\xa0152 pages',
 'isbn': '9798891386211',
 'publisher': '100 Coaches Publishing',
 'publication_date_raw': 'December 2, 2025',
 'cover_url': 'https://www.leadershipnow.com/leadershop/images/9798891386211.jpg',
 'cover_alt_raw': '9798891386211',
 'external_book_url': 'https://amzn.to/3WhDMvU',
 'release_page': '2025 Releases',
 'release_page_url': 'https://www.leadershipnow.com/leadershop/new2025.html',
 'source': 'LeadershipNow / LeaderShop'}

In [116]:
isbn_labels_2025 = test_soup.find_all(
    "b",
    string=lambda text:
        text
        and text.strip() == "ISBN:"
)

print(
    "ISBN labels found:",
    len(isbn_labels_2025)
)

ISBN labels found: 252


In [117]:
records_2025 = []

for isbn_label in isbn_labels_2025:

    record = parse_book_record(
        isbn_label,
        "2025 Releases",
        test_page_url
    )

    if record:
        records_2025.append(
            record
        )

books_2025_df = pd.DataFrame(
    records_2025
)

print(
    "Parsed records:",
    len(books_2025_df)
)

print(
    "Shape:",
    books_2025_df.shape
)

books_2025_df.head()

Parsed records: 252
Shape: (252, 13)


,title,subtitle,author,format_raw,isbn,publisher,publication_date_raw,cover_url,cover_alt_raw,external_book_url,release_page,release_page_url,source
0,The Seismic Shift in You,: The Seven Necessary Shifts to Create Connect...,Michelle Johnston and Marshall Goldsmith,"Hardcover, 152 pages",9798891386211,100 Coaches Publishing,"December 2, 2025",https://www.leadershipnow.com/leadershop/image...,9798891386211,https://amzn.to/3WhDMvU,2025 Releases,https://www.leadershipnow.com/leadershop/new20...,LeadershipNow / LeaderShop
1,The View from Ninety,": Reflections on Living a Long, Contented Life",Charles Handy,"Hardcover, 224 pages",9781529154801,Hutchinson Heinemann,"December 2, 2025",https://www.leadershipnow.com/leadershop/image...,9781529154801,https://amzn.to/4odQXty,2025 Releases,https://www.leadershipnow.com/leadershop/new20...,LeadershipNow / LeaderShop
2,In the Arena,": Theodore Roosevelt in War, Peace, and Revolu...",David S. Brown,"Hardcover, 496 pages",9781668204191,Scribner,"December 2, 2025",https://www.leadershipnow.com/leadershop/image...,9781668204191,https://amzn.to/4q2gAPO,2025 Releases,https://www.leadershipnow.com/leadershop/new20...,LeadershipNow / LeaderShop
3,You to the Power of Two,: Redefining Human Potential in the Age of Ide...,Joseph Bradley and Don Tapscott,"Hardcover, 400 pages",9781637747841,BenBella Books,"December 2, 2025",https://www.leadershipnow.com/leadershop/image...,9781637747841,https://amzn.to/3XqSEc4,2025 Releases,https://www.leadershipnow.com/leadershop/new20...,LeadershipNow / LeaderShop
4,One Move Makes All the Difference,: How to discover your power and transform you...,Martin R. Mendelson,"Paperback, 200 pages",9781636987897,Morgan James Publishing,"December 2, 2025",https://www.leadershipnow.com/leadershop/image...,9781636987897,https://amzn.to/49OQSsD,2025 Releases,https://www.leadershipnow.com/leadershop/new20...,LeadershipNow / LeaderShop


In [118]:
print(
    "Expected records:",
    252
)

print(
    "Parsed records:",
    len(books_2025_df)
)

print(
    "\nMissing values:"
)

print(
    books_2025_df.isna().sum()
)

Expected records: 252
Parsed records: 252

Missing values:
title                    0
subtitle                12
author                   0
format_raw               0
isbn                     0
publisher                0
publication_date_raw     0
cover_url                0
cover_alt_raw            0
external_book_url        0
release_page             0
release_page_url         0
source                   0
dtype: int64


In [119]:
print(
    "Unique displayed ISBNs:",
    books_2025_df[
        "isbn"
    ].nunique()
)

print(
    "Duplicate displayed ISBNs:",
    books_2025_df[
        "isbn"
    ].duplicated().sum()
)

Unique displayed ISBNs: 252
Duplicate displayed ISBNs: 0


In [120]:
isbn_cover_check = (
    books_2025_df[
        "isbn"
    ].astype(str).str.strip()
    ==
    books_2025_df[
        "cover_alt_raw"
    ].astype(str).str.strip()
)

print(
    "ISBN matches cover alt:",
    isbn_cover_check.sum()
)

print(
    "ISBN does not match cover alt:",
    (~isbn_cover_check).sum()
)

ISBN matches cover alt: 252
ISBN does not match cover alt: 0


In [121]:
books_2025_df.loc[
    ~isbn_cover_check,
    [
        "title",
        "author",
        "isbn",
        "cover_alt_raw",
        "cover_url"
    ]
]

,title,author,isbn,cover_alt_raw,cover_url


## 89. Single-Year Parser Validation

The extraction logic was validated against the complete 2025 LeadershipNow release page before expanding collection to additional years.

### Validation Results

- Expected ISBN records: 252
- Successfully parsed records: 252
- Unique displayed ISBNs: 252
- Duplicate ISBNs: 0
- ISBN-to-cover metadata matches: 252
- ISBN-to-cover metadata mismatches: 0

### Missing Data

All major bibliographic fields were complete:

- title
- author
- ISBN
- format
- publisher
- publication date
- cover URL
- external book URL

Twelve records did not contain a subtitle.

These missing subtitles were retained as missing values because the raw collection stage does not impute information absent from the source.

### Quality-Gate Decision

The 2025 parser achieved complete record recovery and consistent bibliographic extraction.

The extraction logic was therefore approved for application to the complete 2022–2025 collection scope.

In [122]:
def scrape_leadershipnow_page(
    page_name,
    page_url,
    headers,
    delay=1
):
    """
    Scrape one LeadershipNow release page.

    Returns:
        records
        log
    """

    start_time = time.time()

    try:

        response = requests.get(
            page_url,
            headers=headers,
            timeout=30
        )

        status_code = response.status_code

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        isbn_labels = soup.find_all(
            "b",
            string=lambda text:
                text
                and text.strip() == "ISBN:"
        )

        records = []

        for isbn_label in isbn_labels:

            record = parse_book_record(
                isbn_label,
                page_name,
                page_url
            )

            if record:
                records.append(record)

        elapsed_time = round(
            time.time() - start_time,
            2
        )

        log = {
            "release_page": page_name,
            "release_page_url": page_url,
            "status_code": status_code,
            "isbn_labels_found": len(
                isbn_labels
            ),
            "records_parsed": len(
                records
            ),
            "response_length": len(
                response.text
            ),
            "elapsed_seconds": elapsed_time,
            "error": None
        }

        time.sleep(delay)

        return records, log

    except Exception as error:

        elapsed_time = round(
            time.time() - start_time,
            2
        )

        log = {
            "release_page": page_name,
            "release_page_url": page_url,
            "status_code": None,
            "isbn_labels_found": 0,
            "records_parsed": 0,
            "response_length": None,
            "elapsed_seconds": elapsed_time,
            "error": str(error)
        }

        return [], log

In [123]:
all_scraped_records = []
scraping_logs = []

for _, row in target_pages_df.iterrows():

    page_name = row["page"]
    page_url = row["url"]

    print(
        f"Scraping {page_name}..."
    )

    records, log = (
        scrape_leadershipnow_page(
            page_name=page_name,
            page_url=page_url,
            headers=headers,
            delay=1
        )
    )

    all_scraped_records.extend(
        records
    )

    scraping_logs.append(
        log
    )

    print(
        f"Completed: "
        f"{len(records)} records"
    )

Scraping 2025 Releases...
Completed: 252 records
Scraping 2024 Releases...
Completed: 280 records
Scraping 2023 Releases...
Completed: 303 records
Scraping 2022 Releases...
Completed: 289 records


In [124]:
leadershipnow_raw_df = pd.DataFrame(
    all_scraped_records
)

scraping_log_df = pd.DataFrame(
    scraping_logs
)

print(
    "Raw records collected:",
    len(leadershipnow_raw_df)
)

print(
    "Dataset shape:",
    leadershipnow_raw_df.shape
)

Raw records collected: 1124
Dataset shape: (1124, 13)


In [125]:
from datetime import datetime, timezone

scraped_at = datetime.now(
    timezone.utc
).isoformat()

leadershipnow_raw_df.insert(
    0,
    "scrape_id",
    range(
        1,
        len(leadershipnow_raw_df) + 1
    )
)

leadershipnow_raw_df[
    "scraped_at"
] = scraped_at

print(
    "Scraped at:",
    scraped_at
)

leadershipnow_raw_df.head()

Scraped at: 2026-09-22T15:33:26.652006+00:00


,scrape_id,title,subtitle,author,format_raw,isbn,publisher,publication_date_raw,cover_url,cover_alt_raw,external_book_url,release_page,release_page_url,source,scraped_at
0,1,The Seismic Shift in You,: The Seven Necessary Shifts to Create Connect...,Michelle Johnston and Marshall Goldsmith,"Hardcover, 152 pages",9798891386211,100 Coaches Publishing,"December 2, 2025",https://www.leadershipnow.com/leadershop/image...,9798891386211,https://amzn.to/3WhDMvU,2025 Releases,https://www.leadershipnow.com/leadershop/new20...,LeadershipNow / LeaderShop,2026-09-22T15:33:26.652006+00:00
1,2,The View from Ninety,": Reflections on Living a Long, Contented Life",Charles Handy,"Hardcover, 224 pages",9781529154801,Hutchinson Heinemann,"December 2, 2025",https://www.leadershipnow.com/leadershop/image...,9781529154801,https://amzn.to/4odQXty,2025 Releases,https://www.leadershipnow.com/leadershop/new20...,LeadershipNow / LeaderShop,2026-09-22T15:33:26.652006+00:00
2,3,In the Arena,": Theodore Roosevelt in War, Peace, and Revolu...",David S. Brown,"Hardcover, 496 pages",9781668204191,Scribner,"December 2, 2025",https://www.leadershipnow.com/leadershop/image...,9781668204191,https://amzn.to/4q2gAPO,2025 Releases,https://www.leadershipnow.com/leadershop/new20...,LeadershipNow / LeaderShop,2026-09-22T15:33:26.652006+00:00
3,4,You to the Power of Two,: Redefining Human Potential in the Age of Ide...,Joseph Bradley and Don Tapscott,"Hardcover, 400 pages",9781637747841,BenBella Books,"December 2, 2025",https://www.leadershipnow.com/leadershop/image...,9781637747841,https://amzn.to/3XqSEc4,2025 Releases,https://www.leadershipnow.com/leadershop/new20...,LeadershipNow / LeaderShop,2026-09-22T15:33:26.652006+00:00
4,5,One Move Makes All the Difference,: How to discover your power and transform you...,Martin R. Mendelson,"Paperback, 200 pages",9781636987897,Morgan James Publishing,"December 2, 2025",https://www.leadershipnow.com/leadershop/image...,9781636987897,https://amzn.to/49OQSsD,2025 Releases,https://www.leadershipnow.com/leadershop/new20...,LeadershipNow / LeaderShop,2026-09-22T15:33:26.652006+00:00


In [126]:
expected_total = (
    target_pages_df[
        "isbn_count"
    ].sum()
)

actual_total = len(
    leadershipnow_raw_df
)

print(
    "Expected raw records:",
    expected_total
)

print(
    "Actual raw records:",
    actual_total
)

print(
    "Difference:",
    actual_total - expected_total
)

Expected raw records: 1124
Actual raw records: 1124
Difference: 0


In [127]:
leadershipnow_raw_df[
    "release_page"
].value_counts()

release_page
2023 Releases    303
2022 Releases    289
2024 Releases    280
2025 Releases    252
Name: count, dtype: int64

In [128]:
missing_summary = pd.DataFrame({
    "missing_count":
        leadershipnow_raw_df
        .isna()
        .sum(),

    "missing_pct":
        (
            leadershipnow_raw_df
            .isna()
            .mean()
            * 100
        ).round(2)
})

missing_summary

,missing_count,missing_pct
scrape_id,0,0.00
title,0,0.00
subtitle,43,3.83
author,0,0.00
format_raw,0,0.00
isbn,0,0.00
publisher,0,0.00
publication_date_raw,0,0.00
cover_url,0,0.00
cover_alt_raw,0,0.00


In [129]:
print(
    "Total records:",
    len(leadershipnow_raw_df)
)

print(
    "Unique ISBN values:",
    leadershipnow_raw_df[
        "isbn"
    ].nunique()
)

print(
    "Duplicate ISBN occurrences:",
    leadershipnow_raw_df[
        "isbn"
    ].duplicated().sum()
)

Total records: 1124
Unique ISBN values: 1120
Duplicate ISBN occurrences: 4


In [130]:
isbn_length_summary = (
    leadershipnow_raw_df[
        "isbn"
    ]
    .astype(str)
    .str.strip()
    .str.len()
    .value_counts()
    .sort_index()
)

isbn_length_summary

isbn
12       2
13    1121
14       1
Name: count, dtype: int64

In [131]:
displayed_isbn = (
    leadershipnow_raw_df[
        "isbn"
    ]
    .astype(str)
    .str.strip()
)

cover_alt = (
    leadershipnow_raw_df[
        "cover_alt_raw"
    ]
    .astype(str)
    .str.strip()
)

cover_available = (
    leadershipnow_raw_df[
        "cover_alt_raw"
    ].notna()
)

cover_match = (
    displayed_isbn
    == cover_alt
)

print(
    "Records with cover metadata:",
    cover_available.sum()
)

print(
    "Matching ISBN / cover alt:",
    (
        cover_available
        & cover_match
    ).sum()
)

print(
    "Non-matching ISBN / cover alt:",
    (
        cover_available
        & ~cover_match
    ).sum()
)

print(
    "Missing cover alt:",
    (~cover_available).sum()
)

Records with cover metadata: 1124
Matching ISBN / cover alt: 1124
Non-matching ISBN / cover alt: 0
Missing cover alt: 0


In [132]:
cover_anomalies_df = (
    leadershipnow_raw_df.loc[
        (~cover_available)
        |
        (
            cover_available
            & ~cover_match
        ),
        [
            "title",
            "author",
            "isbn",
            "cover_alt_raw",
            "cover_url",
            "release_page"
        ]
    ]
)

cover_anomalies_df

,title,author,isbn,cover_alt_raw,cover_url,release_page


In [133]:
duplicate_isbn_df = (
    leadershipnow_raw_df[
        leadershipnow_raw_df[
            "isbn"
        ].duplicated(
            keep=False
        )
    ]
    .sort_values(
        "isbn"
    )
)

print(
    "Rows involved in duplicate ISBNs:",
    len(duplicate_isbn_df)
)

duplicate_isbn_df[
    [
        "title",
        "author",
        "isbn",
        "publication_date_raw",
        "release_page"
    ]
].head(30)

Rows involved in duplicate ISBNs: 8


,title,author,isbn,publication_date_raw,release_page
813,Get It Done,Ayelet Fishbach,9780316538343,"January 4, 2023",2023 Releases
1101,Get It Done,Ayelet Fishbach,9780316538343,"January 4, 2022",2022 Releases
713,Burn Rate,Andy Dunn,9780593238264,"May 10, 2023",2023 Releases
1009,Burn Rate,Andy Dunn,9780593238264,"May 10, 2022",2022 Releases
809,Win Every Argument,Mehdi Hasan,9781250853479,"February 28, 2023",2023 Releases
864,Win Every Argument,Mehdi Hasan,9781250853479,"November 15, 2022",2022 Releases
181,You're the Boss,Sabina Nawaz,9781668023181,"March 4, 2025",2025 Releases
463,You're the Boss,Sabina Nawaz,9781668023181,"March 4, 2024",2024 Releases


In [134]:
leadershipnow_raw_file = (
    raw_data_path
    / "leadershipnow_books_raw.csv"
)

leadershipnow_raw_df.to_csv(
    leadershipnow_raw_file,
    index=False
)

print(
    "Raw dataset saved:"
)

print(
    leadershipnow_raw_file
)

Raw dataset saved:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/raw/leadershipnow_books_raw.csv


In [135]:
leadershipnow_log_file = (
    raw_data_path
    / "leadershipnow_scraping_log.csv"
)

scraping_log_df.to_csv(
    leadershipnow_log_file,
    index=False
)

print(
    "Scraping log saved:"
)

print(
    leadershipnow_log_file
)

Scraping log saved:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/raw/leadershipnow_scraping_log.csv


In [136]:
leadershipnow_anomaly_file = (
    raw_data_path
    / "leadershipnow_cover_anomalies.csv"
)

cover_anomalies_df.to_csv(
    leadershipnow_anomaly_file,
    index=False
)

print(
    "Cover anomaly report saved:"
)

print(
    leadershipnow_anomaly_file
)

Cover anomaly report saved:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/raw/leadershipnow_cover_anomalies.csv


## 99. Final Web-Scraping Results

The production web-scraping stage successfully collected leadership and management book records from LeadershipNow / LeaderShop for four complete release years: 2022–2025.

### Collection Results

| Release Year | Raw Records |
|---|---:|
| 2025 | 252 |
| 2024 | 280 |
| 2023 | 303 |
| 2022 | 289 |
| **Total** | **1,124** |

The expected number of records was 1,124 and the scraper recovered exactly 1,124 records, resulting in zero collection loss.

### Data Completeness

All collected records contain:

- title
- author
- format information
- displayed ISBN
- publisher
- publication date
- cover URL
- cover metadata
- external book URL
- source provenance

The only field containing missing values was `subtitle`.

A total of 43 records (3.83%) did not contain a subtitle on the source page. These values were intentionally retained as missing rather than imputed.

### Identifier Validation

The raw collection contains:

- 1,124 total records
- 1,120 unique displayed ISBN values
- 4 duplicate ISBN occurrences
- 8 rows involved in duplicated ISBNs

ISBN length inspection also identified:

- 1,121 values containing 13 characters
- 2 values containing 12 characters
- 1 value containing 14 characters

These unusual identifiers will be investigated during data cleaning rather than modified during collection.

### Cover Metadata Validation

All 1,124 records contained cover metadata.

The displayed ISBN matched the cover-image `alt` metadata for all 1,124 records.

No cover metadata mismatches were identified within the 2022–2025 production collection.

### Raw-Data Principle

No duplicate records were removed and no missing values, identifiers, dates, titles, subtitles, author names, publishers, or format information were corrected during collection.

The raw dataset therefore preserves the source as collected.

Cleaning and normalization will be performed separately in Notebook 03.

## 100. Duplicate Record Observation

Preliminary validation identified four ISBN values appearing more than once in the raw collection.

Eight rows are involved in these duplicate ISBN groups.

Examples include books appearing across adjacent annual release pages, sometimes with different publication-date values.

These records were not removed during web scraping.

During Notebook 03, duplicate groups will be investigated using:

- ISBN
- title
- author
- publisher
- publication date
- release page

The cleaning stage will determine whether each duplicate represents:

1. the same bibliographic edition repeated by the source;
2. a publication-date update;
3. a source-page carryover;
4. or another metadata inconsistency.

Deduplication decisions will therefore be evidence-based rather than based only on repeated ISBN values.

## 101. Limitations of the Web-Scraped Dataset

Several limitations should be considered when interpreting the collected data.

### Curated Source

LeadershipNow / LeaderShop is a curated leadership-oriented publication source.

The collected books should therefore not be interpreted as a random or exhaustive sample of all leadership and management books published worldwide.

### Time Period

The production collection intentionally uses four complete release years, 2022–2025.

The current 2026 release page was excluded from the production sample because 2026 was incomplete at the time of data collection.

### Source Classification

Books are included according to LeadershipNow's editorial and collection decisions.

Some titles may address leadership indirectly through areas such as:

- organizational behavior
- communication
- strategy
- psychology
- technology
- personal development
- decision making
- workplace performance

Later NLP and clustering analysis will examine the thematic structure of the collection rather than assuming that every book belongs to an identical leadership category.

### Missing Metadata

Not every book contains a subtitle.

Missing source information was preserved rather than artificially generated.

### External Links

External book-information links were collected as source metadata but were not followed or scraped.

### Historical Metadata

Publication information represents the values displayed by LeadershipNow at the time of collection. Historical pages may contain revised dates or repeated titles across release years.

These issues will be investigated during data cleaning and validation.

## 102. Output Files

The web-scraping stage generated the following raw files:

### Primary Dataset

`data/raw/leadershipnow_books_raw.csv`

Contains the 1,124 raw book records collected from the 2022–2025 LeadershipNow release pages.

### Scraping Log

`data/raw/leadershipnow_scraping_log.csv`

Contains page-level collection information used to verify HTTP accessibility and extraction completeness.

### Cover Anomaly Report

`data/raw/leadershipnow_cover_anomalies.csv`

Contains records where displayed ISBN metadata and cover metadata disagree or where cover metadata is missing.

No such anomalies were identified in the final 2022–2025 production collection.

# Conclusion

Notebook 02 successfully completed the web-scraping component of the Leadership and Management Book Recommendation System.

After evaluating multiple potential sources for accessibility, domain relevance, metadata quality, and scalability, LeadershipNow / LeaderShop was selected as the primary production source.

The final production scrape collected 1,124 raw records covering four complete release years from 2022 through 2025.

The collection achieved complete recovery relative to the expected page-level record counts and preserved the source data without cleaning, imputation, deduplication, or normalization.

Together with the Open Library API collection completed in Notebook 01, the project now has two independently collected raw data sources:

- Open Library API: 1,000 raw search records plus work-level enrichment
- LeadershipNow / LeaderShop: 1,124 raw scraped records

These sources provide complementary information. Open Library contributes broader bibliographic, subject, rating, reading-interest, and work-level metadata, while LeadershipNow provides a domain-focused collection of leadership and management titles with consistent ISBN, publisher, publication-date, format, and cover information.

The next stage will clean each source independently before attempting cross-source integration.

# Next Step

Proceed to:

**Notebook 03 — Data Cleaning**

The next notebook will:

1. load all raw API and web-scraped datasets;
2. preserve immutable copies of the raw inputs;
3. assess schema and data types;
4. standardize missing values;
5. clean titles, subtitles, and author fields;
6. validate and standardize ISBN identifiers;
7. investigate the 12- and 14-character ISBN anomalies;
8. parse publication dates and years;
9. separate format and page-count information;
10. investigate duplicate ISBN groups;
11. assess publisher consistency;
12. prepare API and scraped datasets for later integration;
13. produce cleaned datasets without prematurely merging the two sources.